# Quant Terminal v21
### Self-Learning · Options IV Earnings Flag · Autonomous · Continuous Feedback Loop
> Predicts. Tracks outcomes. Diagnoses failures. Rewrites its own rules. Flags earnings uncertainty. Reports every morning.

In [ ]:
# ============================================================
# CELL 1 — INSTALL  (run alone first → Restart → Run all)
# ============================================================
import sys, subprocess

def pip(pkg):
    return subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                          capture_output=True,text=True).returncode

pkgs = [
    "pandas>=2.2","scikit-learn>=1.5","xgboost>=2.1","lightgbm>=4.4",
    "catboost>=1.2.5","optuna>=3.6","yfinance>=0.2.40","arch>=6.3",
    "hmmlearn>=0.3.2","cvxpy>=1.5","ta>=0.11","pandas_ta>=0.3.14b0",
    "requests>=2.31","ipywidgets>=8.1","mapie==0.8.6",
    "imbalanced-learn>=0.12","mlflow>=2.13","matplotlib>=3.9",
    "seaborn>=0.13","threadpoolctl==3.1.0","numba>=0.61",
    "river>=0.21",
]
print("Installing packages...")
for pkg in pkgs:
    r = pip(pkg)
    print(f"  {'OK' if r==0 else 'WARN'} {pkg.split('>=')[0].split('==')[0]}")

pip("torch --index-url https://download.pytorch.org/whl/cpu")
pip("transformers>=4.41")

import numpy as np
print(f"\nNumPy {np.__version__}")
print("\nInstall complete. >>> Runtime -> Restart -> Run all <<<")


In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import os, json, datetime, time, traceback, threading, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests

import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
import cvxpy as cp
from IPython.display import display, HTML, clear_output
import ta

# River online learning
from river import linear_model, preprocessing, metrics, drift

try:
    from mapie.classification import MapieClassifier
    _MAPIE_OK = True
except ImportError:
    class MapieClassifier:
        def __init__(self,estimator=None,**kw): self.estimator=estimator
        def fit(self,X,y): self.estimator.fit(X,y); return self
        def predict(self,X,**kw):
            p=self.estimator.predict_proba(X)[:,1]
            return self.estimator.predict(X),np.stack([1-p,p],axis=1).reshape(len(X),1,2)
    _MAPIE_OK = False

try:
    import torch; torch.manual_seed(42)
except Exception:
    pass

np.random.seed(42)
print(f"Imports OK — pandas {pd.__version__} | numpy {np.__version__} | xgb {xgb.__version__}")
print("River online learning imported OK")


In [ ]:
# ============================================================
# CELL 3 — CONFIGURATION
# ============================================================
ALPACA_API_KEY    = ""
ALPACA_SECRET_KEY = ""
NEWS_API_KEY      = ""
FRED_API_KEY      = ""

# ── UPGRADE: Tradier ($10/month) ────────────────────────────
# When ready to upgrade from Finnhub → Tradier:
#   1. Open a Tradier brokerage account at tradier.com ($10/mo)
#   2. Get API key from dashboard.tradier.com/settings/api
#   3. Uncomment and fill in below:
#
# TRADIER_API_KEY = ""   # paste your Tradier API key here
#
# Tradier gives you:
#   - Verified earnings calendar (cross-referenced against SEC filings)
#   - Real-time options chains with proper bid/ask spreads
#     (replaces yfinance options chain in get_options_iv_flag)
#   - More reliable than yfinance for IV straddle calculation
#   - Combine with Polygon.io ($29/mo) for the full upgrade:
#     Polygon = intraday data | Tradier = options + earnings
#     Total: $39/mo → model score ~73 → ~78
#
# To activate: search this notebook for "TRADIER_UPGRADE"
# and uncomment the relevant sections.
# ─────────────────────────────────────────────────────────────
TRADIER_API_KEY = ""   # leave empty until upgrade



DEFAULT_WATCHLIST = [
    "AAPL","MSFT","NVDA","GOOGL","AMZN","META","TSLA",
    "JPM","V","UNH","SPY",
    "BTC-USD","ETH-USD","SOL-USD","BNB-USD","XRP-USD"
]

FORECAST_DAYS    = 5
TRAIN_START      = "2019-01-01"
TRAIN_END        = datetime.date.today().isoformat()
PORTFOLIO_CAPITAL= 10_000
MIN_CONFIDENCE   = 0.65
MAX_POSITION_PCT = 0.20
STOP_LOSS_PCT    = 0.05
MAX_DRAWDOWN_PCT = 0.15
HMM_STATES       = 3
GARCH_PATHS      = 5_000

MACRO            = {}  # populated by Cell 4 fetch_macro()

PT_LOG_COLS = ["ts","ticker","action","price","qty",
               "confidence","regime","order_id","status"]

# ── Self-learning log columns ────────────────────────────────
PRED_LOG_COLS = [
    "pred_ts","ticker","action","confidence","price_at_pred",
    "p_ensemble","p_up_garch","rsi","regime","vix","yield_curve",
    "ism_pmi","unemployment","sentiment","horizon_days",
    "outcome_ts","price_at_outcome","actual_return",
    "was_correct","magnitude_error","scored",
    "iv_flag","iv_scale","iv_note"
]

# ── Adaptive weight state (River updates these live) ─────────
ADAPTIVE_WEIGHTS = {
    "w_ensemble":  0.55,
    "w_garch":     0.20,
    "w_sentiment": 0.10,
    "w_regime":    0.10,
    "w_yieldcurve":0.05,
}

# ── Learned rule overrides (model writes these itself) ───────
LEARNED_RULES = {}  # e.g. {"high_rsi_bear": {"dampen": 0.15, "count": 7}}

# ── Google Drive ─────────────────────────────────────────────
_drive_mounted = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _drive_dir  = Path("/content/drive/MyDrive/quant_terminal_v21")
    _drive_dir.mkdir(parents=True, exist_ok=True)
    _drive_mounted = True
    print(f"✅ Drive mounted → {_drive_dir}")
except Exception as _e:
    _drive_dir = Path(".")
    print(f"⚠️  Drive not available ({_e}) — session-only mode")

# Sub-directories for organised storage
(_drive_dir / "paper_trades").mkdir(exist_ok=True)
(_drive_dir / "predictions").mkdir(exist_ok=True)
(_drive_dir / "models").mkdir(exist_ok=True)
(_drive_dir / "weights").mkdir(exist_ok=True)

In [ ]:
# ============================================================
# CELL 4 — MACRO DATA
# ============================================================
MACRO: dict = {}

def _yf_latest(ticker, period="5d"):
    try:
        df = yf.Ticker(ticker).history(period=period, auto_adjust=True)
        if df.empty: return None
        return float(df["Close"].dropna().iloc[-1])
    except Exception: return None

def _fred(series_id, fallback=None):
    if not FRED_API_KEY: return fallback
    try:
        url = (f"https://api.stlouisfed.org/fred/series/observations"
               f"?series_id={series_id}&api_key={FRED_API_KEY}"
               f"&file_type=json&sort_order=desc&limit=2")
        r = requests.get(url, timeout=8)
        for o in r.json().get("observations",[]):
            try: return float(o["value"])
            except Exception: continue
        return fallback
    except Exception: return fallback

def fetch_macro():
    global MACRO
    fed_rate  = _yf_latest("^IRX")
    tnx_10y   = _yf_latest("^TNX")
    irx_2y    = _yf_latest("^IRX")
    vix       = _yf_latest("^VIX")
    dxy       = _yf_latest("DX-Y.NYB")
    wti_crude = _yf_latest("CL=F")
    gold      = _yf_latest("GC=F")
    hyg       = _yf_latest("HYG")
    lqd       = _yf_latest("LQD")
    credit_spread = round(hyg/lqd,4) if hyg and lqd else None
    spy_info  = {}
    try: spy_info = yf.Ticker("SPY").info
    except Exception: pass
    spy_pe     = spy_info.get("trailingPE")
    ey         = round(100/spy_pe,2) if spy_pe and spy_pe>0 else None
    ey_spread  = round(ey-(tnx_10y or 4.3),2) if ey else None
    yc         = round((tnx_10y or 4.3)-(irx_2y or 5.0),3)
    unemployment = _fred("UNRATE",   fallback=3.9)
    cpi_yoy      = _fred("CPIAUCSL", fallback=3.1)
    gdp_growth   = _fred("A191RL1Q225SBEA", fallback=2.8)
    retail_sales = _fred("RSAFS",    fallback=0.4)
    ism_pmi      = _fred("MANEMP",   fallback=50.3)
    crypto_fg = None; crypto_fg_label = "unavailable"
    try:
        r = requests.get("https://api.alternative.me/fng/?limit=1",timeout=5)
        crypto_fg       = int(r.json()["data"][0]["value"])
        crypto_fg_label = r.json()["data"][0]["value_classification"]
    except Exception: pass
    if vix:
        if vix<15:    mr,mc="Risk-on / Bull","success"
        elif vix<25:  mr,mc="Neutral / Mixed","warning"
        else:         mr,mc="Risk-off / Bear","danger"
    else: mr,mc="Unknown","secondary"
    MACRO = dict(
        fed_rate=fed_rate,tnx_10y=tnx_10y,yield_curve=yc,
        vix=vix,dxy=dxy,wti_crude=wti_crude,gold=gold,
        credit_spread=credit_spread,earnings_yield=ey,ey_spread=ey_spread,
        unemployment=unemployment,cpi_yoy=cpi_yoy,gdp_growth=gdp_growth,
        retail_sales=retail_sales,ism_pmi=ism_pmi,
        crypto_fg=crypto_fg,crypto_fg_label=crypto_fg_label,
        macro_regime=mr,macro_regime_color=mc
    )
    return MACRO

print("Fetching macro data...")
fetch_macro()
for k,v in MACRO.items():
    if v is not None and k not in ("macro_regime","macro_regime_color","crypto_fg_label"):
        print(f"  {k:<20} {v}")
print(f"\nMacro ready | regime: {MACRO['macro_regime']}")


In [ ]:
# ============================================================
# CELL 5 — TICKER DATA DOWNLOAD
# ============================================================
def download_ticker(ticker, start, end):
    for attempt in range(3):
        try:
            df = yf.Ticker(ticker).history(start=start,end=end,auto_adjust=True)
            if df.empty: return None
            df.index = pd.to_datetime(df.index).tz_localize(None)
            df = df[["Open","High","Low","Close","Volume"]].copy()
            df.dropna(subset=["Close"],inplace=True)
            return df
        except Exception as e:
            if attempt==2: print(f"  {ticker}: {e}")
            time.sleep(1)
    return None

raw_data = {}
print(f"Downloading {len(DEFAULT_WATCHLIST)} tickers...")
for tk in DEFAULT_WATCHLIST:
    df = download_ticker(tk,TRAIN_START,TRAIN_END)
    if df is not None and len(df)>100:
        raw_data[tk] = df
        print(f"  OK {tk:<10} {len(df)} rows  ${df['Close'].iloc[-1]:.2f}")
    else:
        print(f"  SKIP {tk}")
print(f"\n{len(raw_data)}/{len(DEFAULT_WATCHLIST)} tickers ready")


In [ ]:
# ============================================================
# CELL 6 — FEATURE ENGINEERING  (macro-aware)
# ============================================================
def build_features(df):
    d = df.copy()
    c,h,l,v = d["Close"],d["High"],d["Low"],d["Volume"]
    for k in [1,3,5,10,21]: d[f"ret_{k}d"]=c.pct_change(k)
    for w in [5,10,20,50,200]:
        d[f"sma_{w}"]=c.rolling(w).mean()
        d[f"sma_r_{w}"]=c/d[f"sma_{w}"]-1
    d["ema_12"]=c.ewm(span=12,adjust=False).mean()
    d["ema_26"]=c.ewm(span=26,adjust=False).mean()
    d["macd"]=d["ema_12"]-d["ema_26"]
    d["macd_sig"]=d["macd"].ewm(span=9,adjust=False).mean()
    d["macd_h"]=d["macd"]-d["macd_sig"]
    d["rsi_14"]=ta.momentum.rsi(c,window=14)
    d["rsi_7"]=ta.momentum.rsi(c,window=7)
    stoch=ta.momentum.StochasticOscillator(h,l,c)
    d["stoch_k"]=stoch.stoch(); d["stoch_d"]=stoch.stoch_signal()
    d["cci"]=ta.trend.CCIIndicator(h,l,c).cci()
    d["willr"]=ta.momentum.WilliamsRIndicator(h,l,c).williams_r()
    d["mfi"]=ta.volume.MFIIndicator(h,l,c,v).money_flow_index()
    bb=ta.volatility.BollingerBands(c)
    d["bb_upper"]=bb.bollinger_hband(); d["bb_lower"]=bb.bollinger_lband()
    d["bb_pct"]=bb.bollinger_pband(); d["bb_w"]=bb.bollinger_wband()
    d["atr_14"]=ta.volatility.AverageTrueRange(h,l,c).average_true_range()
    kelt=ta.volatility.KeltnerChannel(h,l,c)
    d["kelt_u"]=kelt.keltner_channel_hband()
    d["kelt_l"]=kelt.keltner_channel_lband()
    d["vol_r_20"]=v/v.rolling(20).mean()
    d["obv"]=ta.volume.OnBalanceVolumeIndicator(c,v).on_balance_volume()
    d["vwap"]=(c*v).cumsum()/v.cumsum()
    d["dow"]=d.index.dayofweek; d["month"]=d.index.month
    d["body"]=(c-d["Open"]).abs()/(h-l+1e-9)
    d["upper_w"]=(h-c.clip(lower=d["Open"]))/(h-l+1e-9)
    d["lower_w"]=(c.clip(upper=d["Open"])-l)/(h-l+1e-9)
    for w in [5,10,21]: d[f"rvol_{w}"]=d["ret_1d"].rolling(w).std()*np.sqrt(252)
    macro_vals = {
        "m_vix":    MACRO.get("vix") or 20.0,
        "m_tnx":    MACRO.get("tnx_10y") or 4.3,
        "m_yc":     MACRO.get("yield_curve") or 0.0,
        "m_dxy":    MACRO.get("dxy") or 104.0,
        "m_wti":    MACRO.get("wti_crude") or 80.0,
        "m_gold":   MACRO.get("gold") or 2000.0,
        "m_credit": MACRO.get("credit_spread") or 1.0,
        "m_unemp":  MACRO.get("unemployment") or 4.0,
        "m_cpi":    MACRO.get("cpi_yoy") or 3.0,
        "m_gdp":    MACRO.get("gdp_growth") or 2.5,
        "m_pmi":    MACRO.get("ism_pmi") or 50.0,
        "m_cfg":    float(MACRO.get("crypto_fg") or 50),
    }
    for col,val in macro_vals.items(): d[col]=val
    d["target"]=(c.shift(-FORECAST_DAYS)>c).astype(int)
    feat_cols=[col for col in d.columns
               if col not in ["Open","High","Low","Close","Volume","target"]]
    d[feat_cols]=d[feat_cols].shift(1)
    d.dropna(inplace=True)
    return d

print("Building features...")
featured = {}
for tk,df in raw_data.items():
    fd=build_features(df)
    if len(fd)>200:
        featured[tk]=fd
        print(f"  OK {tk:<10} {len(fd)} rows  {len(fd.columns)} features")

FEATURE_COLS=[c for c in next(iter(featured.values())).columns
              if c not in ["Open","High","Low","Close","Volume","target"]]
print(f"\n{len(featured)} tickers | {len(FEATURE_COLS)} features (incl macro)")


In [ ]:
# ============================================================
# CELL 7 — HMM REGIME DETECTION
# ============================================================
def fit_hmm(df, n_states=HMM_STATES):
    returns=df["Close"].pct_change().dropna().values.reshape(-1,1)
    model=GaussianHMM(n_components=n_states,covariance_type="full",
                      n_iter=200,random_state=42)
    model.fit(returns)
    labels=model.predict(returns)
    s=pd.Series(labels,index=df.index[1:],name="regime")
    return s.reindex(df.index).ffill().fillna(0).astype(int)

print("Fitting HMM regimes...")
regimes = {}
for tk,df in featured.items():
    regimes[tk]=fit_hmm(raw_data[tk].loc[df.index[0]:])
    print(f"  OK {tk:<10} regime={regimes[tk].iloc[-1]}")
print("\nRegimes complete")


In [ ]:
# ============================================================
# CELL 8 — ML ENSEMBLE TRAINING
# ============================================================
def train_ensemble(df, ticker):
    X=df[FEATURE_COLS].values; y=df["target"].values
    scaler=StandardScaler(); X_sc=scaler.fit_transform(X)
    tscv=TimeSeriesSplit(n_splits=5)
    def xgb_obj(trial):
        p=dict(
            n_estimators=trial.suggest_int("n",100,500),
            max_depth=trial.suggest_int("d",3,9),
            learning_rate=trial.suggest_float("lr",1e-3,0.3,log=True),
            subsample=trial.suggest_float("sub",0.5,1.0),
            colsample_bytree=trial.suggest_float("col",0.5,1.0),
            gamma=trial.suggest_float("g",0,5),
            reg_alpha=trial.suggest_float("a",0,3),
            reg_lambda=trial.suggest_float("l",0,3),
            use_label_encoder=False,eval_metric="logloss",
            tree_method="hist",random_state=42,verbosity=0)
        aucs=[]
        for ti,vi in tscv.split(X_sc):
            Xtr,Xva,ytr,yva=X_sc[ti],X_sc[vi],y[ti],y[vi]
            try: Xtr,ytr=SMOTE(random_state=42).fit_resample(Xtr,ytr)
            except Exception: pass
            m=xgb.XGBClassifier(**p)
            m.fit(Xtr,ytr,eval_set=[(Xva,yva)],verbose=False)
            if len(np.unique(yva))>1:
                aucs.append(roc_auc_score(yva,m.predict_proba(Xva)[:,1]))
        return np.mean(aucs) if aucs else 0.5
    study=optuna.create_study(direction="maximize")
    study.optimize(xgb_obj,n_trials=25,show_progress_bar=False)
    bp=study.best_params
    best_xgb=xgb.XGBClassifier(
        n_estimators=bp["n"],max_depth=bp["d"],learning_rate=bp["lr"],
        subsample=bp["sub"],colsample_bytree=bp["col"],gamma=bp["g"],
        reg_alpha=bp["a"],reg_lambda=bp["l"],
        use_label_encoder=False,eval_metric="logloss",
        tree_method="hist",random_state=42,verbosity=0)
    best_lgb=lgb.LGBMClassifier(
        n_estimators=400,max_depth=6,learning_rate=0.05,
        num_leaves=63,subsample=0.8,colsample_bytree=0.8,
        random_state=42,verbose=-1)
    best_cat=CatBoostClassifier(
        iterations=300,depth=6,learning_rate=0.05,random_seed=42,verbose=0)
    try: X_r,y_r=SMOTE(random_state=42).fit_resample(X_sc,y)
    except Exception: X_r,y_r=X_sc,y
    best_xgb.fit(X_r,y_r); best_lgb.fit(X_r,y_r); best_cat.fit(X_r,y_r)
    n_val=max(int(len(X_sc)*0.2),30)
    Xva,yva=X_sc[-n_val:],y[-n_val:]
    prob=(best_xgb.predict_proba(Xva)[:,1]+
          best_lgb.predict_proba(Xva)[:,1]+
          best_cat.predict_proba(Xva)[:,1])/3
    auc=roc_auc_score(yva,prob) if len(np.unique(yva))>1 else 0.5
    fi=pd.Series(best_xgb.feature_importances_,
                 index=FEATURE_COLS).sort_values(ascending=False)
    return dict(xgb=best_xgb,lgb=best_lgb,cat=best_cat,
                scaler=scaler,auc=auc,fi=fi)

print("Training ensembles (10-20 min first run)...")
models = {}
for tk,df in featured.items():
    print(f"  {tk}...",end=" ",flush=True)
    try:
        m=train_ensemble(df,tk); models[tk]=m
        print(f"AUC={m['auc']:.3f} OK")
    except Exception as e:
        print(f"FAILED: {e}")
print(f"\n{len(models)}/{len(featured)} trained")


In [ ]:

# ============================================================
# OPTIONS IV EARNINGS FLAG  (free via yfinance)
# ============================================================
# Fetches implied volatility from options chain before earnings.
# Returns: expected_move_pct, is_earnings_week, iv_flag
# Used in generate_signal to reduce confidence on high-IV names.

# ============================================================
# HARDENED EARNINGS DATE DETECTION — 3-source + full normaliser
# ============================================================
# Layer 1: Cross-references yfinance (3 methods), Earnings Whispers,
#           and Alpha Vantage. Trusts a date only when 2+ sources agree.
# Layer 2: Normalises every timestamp format (UNIX int, string,
#           Timestamp, datetime) to a timezone-naive date object.
# Layer 3: Validates business-sense rules (weekday, 0-120 days out).

def _normalise_earnings_date(raw) -> "datetime.date | None":
    """
    Convert any earnings date format to a timezone-naive datetime.date.
    Handles: UNIX int, float, ISO string, pd.Timestamp, datetime.datetime.
    Returns None if conversion fails or result fails sanity checks.
    """
    import datetime as _dt
    import pandas as _pd
    try:
        if raw is None:
            return None

        # UNIX timestamp (int or float)
        if isinstance(raw, (int, float)) and not isinstance(raw, bool):
            if raw > 1_000_000_000:   # looks like UNIX seconds
                d = _dt.datetime.utcfromtimestamp(raw).date()
            elif raw > 1_000_000:     # maybe UNIX milliseconds
                d = _dt.datetime.utcfromtimestamp(raw / 1000).date()
            else:
                return None

        # Pandas Timestamp
        elif isinstance(raw, _pd.Timestamp):
            if raw.tzinfo is not None:
                raw = raw.tz_convert("UTC").tz_localize(None)
            d = raw.date()

        # Python datetime
        elif isinstance(raw, _dt.datetime):
            d = raw.date()

        # Python date
        elif isinstance(raw, _dt.date):
            d = raw

        # String
        elif isinstance(raw, str):
            raw = raw.strip()
            if not raw or raw.lower() in ("none","nan","nat","n/a"):
                return None
            # Try common formats
            for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y",
                        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S"):
                try:
                    d = _dt.datetime.strptime(raw[:len(fmt)], fmt).date()
                    break
                except (ValueError, IndexError):
                    continue
            else:
                # Last resort: pandas parser
                d = _pd.Timestamp(raw).date()

        else:
            return None

        # ── Layer 3 sanity checks ────────────────────────────
        today = _dt.date.today()
        days_out = (d - today).days

        if days_out < 0:
            return None          # in the past — stale data
        if days_out > 120:
            return None          # too far out — likely wrong year
        if d.weekday() > 4:
            return None          # weekend — earnings never on Sat/Sun

        return d

    except Exception:
        return None


def get_earnings_date(ticker: str) -> dict:
    """
    Multi-source earnings date detection with consensus validation.

    Sources tried (in parallel):
      A: yfinance calendar (3 sub-methods)
      B: Earnings Whispers free JSON endpoint
      C: Alpha Vantage earnings calendar (free, 25 req/day)

    Returns:
      {
        "date": datetime.date | None,
        "days_to": int | None,
        "confidence": "HIGH" | "LOW" | "NONE",
        "sources": [list of source names that agreed],
        "is_earnings_week": bool,
        "note": str
      }

    Confidence rules:
      HIGH  — 2+ sources agree within 2-day window
      LOW   — only 1 source returned a date
      NONE  — no source returned a valid date
    """
    import datetime as _dt
    import pandas as _pd
    import requests as _req

    today  = _dt.date.today()
    result = dict(
        date=None, days_to=None,
        confidence="NONE", sources=[],
        is_earnings_week=False, note=""
    )

    candidates = {}   # source_name -> datetime.date

    tk_obj = yf.Ticker(ticker)

    # ── Source A-1: tk.calendar dict ────────────────────────
    try:
        cal = tk_obj.calendar
        if isinstance(cal, dict):
            for key in ("Earnings Date", "earningsDate", "earnings_date"):
                val = cal.get(key)
                if val is None:
                    continue
                # May be a list or scalar
                vals = list(val) if hasattr(val, "__iter__") and not isinstance(val, str) else [val]
                for v in vals:
                    d = _normalise_earnings_date(v)
                    if d:
                        candidates["yf_calendar_dict"] = d
                        break
                if "yf_calendar_dict" in candidates:
                    break
            # Also try earningsTimestamp directly
            ts = cal.get("earningsTimestamp") or cal.get("earningsCallTimestampStart")
            if ts and "yf_calendar_dict" not in candidates:
                d = _normalise_earnings_date(ts)
                if d:
                    candidates["yf_calendar_ts"] = d
    except Exception:
        pass

    # ── Source A-2: tk.calendar DataFrame ───────────────────
    try:
        cal = tk_obj.calendar
        if hasattr(cal, "columns"):
            for col in ("Earnings Date", "earningsDate"):
                if col in cal.columns:
                    raw_vals = cal[col].dropna()
                    for v in raw_vals:
                        d = _normalise_earnings_date(v)
                        if d:
                            candidates["yf_calendar_df"] = d
                            break
                    if "yf_calendar_df" in candidates:
                        break
    except Exception:
        pass

    # ── Source A-3: tk.earnings_dates ───────────────────────
    try:
        ed = tk_obj.earnings_dates
        if ed is not None and not ed.empty:
            # Filter to future dates
            future_mask = ed.index > _pd.Timestamp.now()
            future = ed[future_mask]
            if not future.empty:
                d = _normalise_earnings_date(future.index[0])
                if d:
                    candidates["yf_earnings_dates"] = d
    except Exception:
        pass

    # ── Source A-4: tk.info fields ──────────────────────────
    try:
        info = tk_obj.info
        for field in ("nextEarningsDate", "earningsDate", "earningsTimestamp"):
            val = info.get(field)
            if val:
                d = _normalise_earnings_date(val)
                if d:
                    candidates["yf_info"] = d
                    break
    except Exception:
        pass

    # ── Source B: Earnings Whispers free endpoint ────────────
    try:
        ew_url = f"https://www.earningswhispers.com/api/earningscalendar?ticker={ticker.upper()}"
        r = _req.get(ew_url, timeout=5,
                     headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200:
            data = r.json()
            raw_date = (data.get("epsdatetime") or
                        data.get("earningsdate") or
                        data.get("date"))
            if raw_date:
                d = _normalise_earnings_date(raw_date)
                if d:
                    candidates["earnings_whispers"] = d
    except Exception:
        pass

    # ── Source C: Alpha Vantage earnings calendar (free) ─────
    # Only attempt if FRED_API_KEY available (to avoid burning
    # free AV quota unnecessarily — use same key slot or skip)
    try:
        av_key = "demo"   # demo key works for calendar endpoint
        av_url = (f"https://www.alphavantage.co/query"
                  f"?function=EARNINGS_CALENDAR&symbol={ticker.upper()}"
                  f"&horizon=3month&apikey={av_key}")
        r = _req.get(av_url, timeout=6)
        if r.status_code == 200 and r.text.strip():
            # Returns CSV: symbol,name,reportDate,fiscalDateEnding,...
            lines = r.text.strip().splitlines()
            for line in lines[1:]:   # skip header
                parts = line.split(",")
                if len(parts) >= 3 and parts[0].strip().upper() == ticker.upper():
                    d = _normalise_earnings_date(parts[2].strip())
                    if d:
                        candidates["alpha_vantage"] = d
                        break
    except Exception:
        pass

    # ── Source D: Finnhub earnings calendar (free, no key needed) ──
    # Completely independent of Yahoo Finance — own aggregated feed.
    # Free tier: 60 calls/minute, no API key required.
    try:
        fh_from = today.isoformat()
        fh_to   = (_dt.date.today() + _dt.timedelta(days=90)).isoformat()
        fh_url  = (f"https://finnhub.io/api/v1/calendar/earnings"
                   f"?from={fh_from}&to={fh_to}"
                   f"&symbol={ticker.upper()}&token=")
        r = _req.get(fh_url, timeout=6,
                     headers={"User-Agent": "Mozilla/5.0",
                               "X-Finnhub-Token": ""})
        if r.status_code == 200:
            data = r.json()
            earnings_list = data.get("earningsCalendar", [])
            for item in earnings_list:
                raw_date = item.get("date") or item.get("reportDate")
                if raw_date:
                    d = _normalise_earnings_date(raw_date)
                    if d:
                        candidates["finnhub"] = d
                        break
    except Exception:
        pass


    # ── Consensus logic: find agreement ─────────────────────
    if not candidates:
        result["note"] = "no earnings date found from any source"
        return result

    # Group candidates by date (within 2-day window = same date)
    date_votes: dict = {}
    for source, d in candidates.items():
        placed = False
        for anchor in list(date_votes.keys()):
            if abs((d - anchor).days) <= 2:
                date_votes[anchor].append((source, d))
                placed = True
                break
        if not placed:
            date_votes[d] = [(source, d)]

    # Pick the group with most votes; break ties by earliest date
    best_group = max(date_votes.values(), key=lambda g: (len(g), -g[0][1].toordinal()))
    best_sources = [s for s,_ in best_group]
    best_dates   = [d for _,d in best_group]
    # Use the median date in the group
    best_dates.sort()
    best_date = best_dates[len(best_dates)//2]

    confidence = "HIGH" if len(best_sources) >= 2 else "LOW"

    days_to = (best_date - today).days
    is_ew   = 0 <= days_to <= 7

    result.update(dict(
        date=best_date,
        days_to=days_to,
        confidence=confidence,
        sources=best_sources,
        is_earnings_week=is_ew,
        note=(f"Earnings {days_to}d away ({best_date}) "
              f"[conf={confidence}, sources={best_sources}]")
    ))

    return result


def get_options_iv_flag(ticker: str) -> dict:
    """
    Options IV earnings flag — hardened v21.
    Uses get_earnings_date() for multi-source consensus earnings detection.
    Uses straddle with bid/ask validation + impliedVolatility fallback.
    """
    import datetime as _dt
    import pandas as _pd

    result = dict(
        expected_move_pct=None,
        is_earnings_week=False,
        iv_flag="NORMAL",
        position_scale=1.0,
        ok=False,
        note=""
    )
    try:
        tk_obj = yf.Ticker(ticker)

        # ── Earnings date (multi-source consensus) ────────────
        ed_result = get_earnings_date(ticker)
        result["is_earnings_week"] = ed_result["is_earnings_week"]
        if ed_result["is_earnings_week"]:
            result["note"] += ed_result["note"] + " "

        # ── Options chain ─────────────────────────────────────
        expirations = []
        try:
            expirations = tk_obj.options or []
        except Exception:
            pass

        if not expirations:
            result["note"] += "no options chain"
            if result["is_earnings_week"]:
                result["iv_flag"]        = "ELEVATED"
                result["position_scale"] = 0.50
                result["note"] += " — ELEVATED applied without IV data"
                result["ok"] = True
            return result

        # Nearest expiry >= 5 days out
        today = _dt.date.today()
        near_exp = None
        for exp in expirations:
            try:
                if (_dt.date.fromisoformat(exp) - today).days >= 5:
                    near_exp = exp; break
            except Exception: continue
        if near_exp is None:
            near_exp = expirations[0]

        try:
            chain = tk_obj.option_chain(near_exp)
            calls = chain.calls.copy()
            puts  = chain.puts.copy()
        except Exception as e:
            result["note"] += f"chain error: {str(e)[:40]}"
            if result["is_earnings_week"]:
                result["iv_flag"]        = "ELEVATED"
                result["position_scale"] = 0.50
                result["ok"] = True
            return result

        if calls.empty or puts.empty:
            result["note"] += "empty chain"
            return result

        try:
            hist = tk_obj.history(period="1d", auto_adjust=True)
            spot = float(hist["Close"].iloc[-1]) if not hist.empty else 0.0
        except Exception:
            spot = 0.0

        if spot <= 0:
            result["note"] += "no spot price"
            return result

        # ── ATM straddle with bid/ask validation ──────────────
        exp_move_pct = None
        method_used  = "none"

        try:
            calls["dist"] = (calls["strike"] - spot).abs()
            puts["dist"]  = (puts["strike"]  - spot).abs()
            atm_c = calls.nsmallest(1,"dist").iloc[0]
            atm_p = puts.nsmallest(1,"dist").iloc[0]
            c_bid = float(atm_c.get("bid",0) or 0)
            c_ask = float(atm_c.get("ask",0) or 0)
            p_bid = float(atm_p.get("bid",0) or 0)
            p_ask = float(atm_p.get("ask",0) or 0)
            if c_ask > 0 and c_ask >= c_bid >= 0 and p_ask > 0 and p_ask >= p_bid >= 0:
                straddle = (c_bid+c_ask)/2 + (p_bid+p_ask)/2
                if straddle > 0:
                    exp_move_pct = round(straddle/spot*100, 2)
                    method_used  = "atm_straddle"
        except Exception:
            pass

        # ── impliedVolatility fallback ────────────────────────
        if exp_move_pct is None:
            try:
                calls["dist"] = (calls["strike"] - spot).abs()
                atm_rows = calls.nsmallest(3,"dist")
                iv_vals  = atm_rows["impliedVolatility"].dropna()
                iv_vals  = iv_vals[iv_vals > 0]
                if len(iv_vals) > 0:
                    avg_iv = float(iv_vals.mean())
                    try: dte = max((_dt.date.fromisoformat(near_exp)-today).days, 1)
                    except Exception: dte = 30
                    exp_move_pct = round(avg_iv*(dte/252)**0.5*100, 2)
                    method_used  = "iv_col_fallback"
            except Exception:
                pass

        if exp_move_pct is None:
            result["note"] += "cannot compute expected move"
            if result["is_earnings_week"]:
                result["iv_flag"]        = "ELEVATED"
                result["position_scale"] = 0.50
                result["ok"] = True
            return result

        result["expected_move_pct"] = exp_move_pct
        result["note"] += f"±{exp_move_pct:.1f}% via {method_used}"

        # Classify
        if   exp_move_pct >= 8.0: result["iv_flag"],result["position_scale"] = "HIGH",    0.30
        elif exp_move_pct >= 5.0: result["iv_flag"],result["position_scale"] = "ELEVATED", 0.55
        elif exp_move_pct >= 3.0: result["iv_flag"],result["position_scale"] = "MODERATE", 0.80
        else:                     result["iv_flag"],result["position_scale"] = "NORMAL",   1.0

        # Earnings penalty
        if result["is_earnings_week"]:
            result["position_scale"] = round(result["position_scale"]*0.5, 2)
            result["note"] += " [EARNINGS — position halved]"

        result["ok"] = True

    except Exception as e:
        result["note"] = f"IV error: {str(e)[:80]}"

    return result


def garch_vol_forecast(df, ticker, n_paths=GARCH_PATHS, horizon=FORECAST_DAYS):
    rets=np.log(df["Close"]/df["Close"].shift(1)).dropna()*100
    seed=abs(hash(ticker))%(2**31)
    try:
        res=arch_model(rets,vol="GARCH",p=1,q=1,dist="normal").fit(
            disp="off",show_warning=False)
        fc=res.forecast(horizon=horizon,reindex=False)
        v1d=float(np.sqrt(fc.variance.values[-1,0]))/100
        rng=np.random.default_rng(seed)
        paths=rng.normal(0,rets.std()/100,(n_paths,horizon))
        cr=paths.sum(axis=1)
        p_up=float((cr>0).mean())
        var95=float(np.percentile(cr,5))
        es95=float(cr[cr<=var95].mean()) if (cr<=var95).any() else var95
        return dict(vol1d=v1d,annvol=v1d*np.sqrt(252),
                    p_up=p_up,var95=var95,es95=es95,ok=True)
    except Exception as e:
        return dict(vol1d=0.02,annvol=0.32,p_up=0.5,
                    var95=-0.05,es95=-0.08,ok=False,err=str(e))

print("Running GARCH...")
garch_res = {}
for tk,df in featured.items():
    garch_res[tk]=garch_vol_forecast(df,tk)
    g=garch_res[tk]
    print(f"  {'OK' if g['ok'] else 'WN'} {tk:<10} annvol={g['annvol']:.1%} p_up={g['p_up']:.1%}")
print("\nGARCH complete")

# ── OPTIONS IV FLAGS ─────────────────────────────────────────
print("Fetching options IV flags...")
iv_flags: dict = {}
for tk in featured:
    iv_flags[tk] = get_options_iv_flag(tk)
    f = iv_flags[tk]
    flag_str = f.get("iv_flag","N/A")
    note_str = f.get("note","")
    earn_str = " [EARNINGS]" if f.get("is_earnings_week") else ""
    print(f"  {tk:<10} {flag_str:<10} scale={f.get('position_scale',1.0):.2f}  {note_str[:50]}{earn_str}")
print(f"\nIV flags ready for {len(iv_flags)} tickers")



In [ ]:
# ============================================================
# CELL 10 — FINBERT SENTIMENT
# ============================================================
def fetch_headlines(ticker, n=10):
    if not NEWS_API_KEY: return []
    try:
        url=(f"https://newsapi.org/v2/everything?q={ticker}"
             f"&language=en&pageSize={n}&sortBy=publishedAt"
             f"&apiKey={NEWS_API_KEY}")
        r=requests.get(url,timeout=5)
        return [a.get("title","") for a in r.json().get("articles",[])]
    except Exception: return []

def sentiment_score(headlines):
    if not headlines: return 0.0
    try:
        from transformers import pipeline
        pipe=pipeline("text-classification",model="ProsusAI/finbert",
                      truncation=True,max_length=128)
        lmap={"positive":1,"negative":-1,"neutral":0}
        scores=[lmap.get(pipe(h)[0]["label"].lower(),0)*pipe(h)[0]["score"]
                for h in headlines[:8] if h.strip()]
        return float(np.mean(scores)) if scores else 0.0
    except Exception: return 0.0

print("Computing sentiment...")
sentiments = {}
for tk in featured:
    hl=fetch_headlines(tk)
    sentiments[tk]=sentiment_score(hl)
    src=f"{len(hl)} headlines" if hl else "no key"
    print(f"  OK {tk:<10} {sentiments[tk]:+.3f} ({src})")
print("\nSentiment complete")


In [ ]:
# ============================================================
# CELL 11 — ADAPTIVE SIGNAL GENERATOR
# ============================================================
# Uses ADAPTIVE_WEIGHTS (updated by River) and LEARNED_RULES
# (written by the failure diagnosis engine) to generate signals.

def apply_learned_rules(ticker, rsi, regime, vix, yc, base_composite):
    """Apply self-written rule overrides to dampen or boost confidence."""
    dampener = 1.0
    rules_applied = []

    # High RSI in Bear regime
    if rsi > 68 and regime == 0:
        rule = LEARNED_RULES.get("high_rsi_bear", {})
        if rule.get("count", 0) >= 3:
            dampener *= (1 - rule.get("dampen", 0.0))
            rules_applied.append(f"high_rsi_bear (dampen {rule['dampen']:.0%})")

    # VIX spike rule
    if vix and vix > 28:
        rule = LEARNED_RULES.get("vix_spike", {})
        if rule.get("count", 0) >= 3:
            dampener *= (1 - rule.get("dampen", 0.0))
            rules_applied.append(f"vix_spike (dampen {rule['dampen']:.0%})")

    # Inverted yield curve rule
    if yc and yc < -0.1:
        rule = LEARNED_RULES.get("inverted_yc", {})
        if rule.get("count", 0) >= 3:
            dampener *= (1 - rule.get("dampen", 0.0))
            rules_applied.append(f"inverted_yc (dampen {rule['dampen']:.0%})")

    # Ticker-specific rule
    tk_rule_key = f"ticker_{ticker}"
    rule = LEARNED_RULES.get(tk_rule_key, {})
    if rule.get("count", 0) >= 5:
        dampener *= (1 - rule.get("dampen", 0.0))
        rules_applied.append(f"{tk_rule_key} (dampen {rule['dampen']:.0%})")

    return dampener, rules_applied

def generate_signal(ticker, model_pack, df_feat, regime, garch, sent, iv_flag=None):
    row  = df_feat[FEATURE_COLS].iloc[[-1]].values
    Xsc  = model_pack["scaler"].transform(row)
    p_xgb = float(model_pack["xgb"].predict_proba(Xsc)[0,1])
    p_lgb = float(model_pack["lgb"].predict_proba(Xsc)[0,1])
    p_cat = float(model_pack["cat"].predict_proba(Xsc)[0,1])
    p_ens = (p_xgb+p_lgb+p_cat)/3.0

    regime_score = {0:0.4,1:0.6,2:0.5}.get(regime,0.5)
    sent_norm    = (sent+1)/2
    yc           = MACRO.get("yield_curve") or 0
    yc_score     = 1.0 if yc > 0 else 0.4

    # Use adaptive weights (updated by River learning)
    w = ADAPTIVE_WEIGHTS
    composite = (
        w["w_ensemble"]   * p_ens +
        w["w_garch"]      * garch["p_up"] +
        w["w_sentiment"]  * sent_norm +
        w["w_regime"]     * regime_score +
        w["w_yieldcurve"] * yc_score
    )

    # VIX macro dampener
    vix_val = MACRO.get("vix") or 20
    if vix_val > 30:   composite *= 0.85
    elif vix_val > 22: composite *= 0.93

    # Apply learned rule overrides
    rsi = float(df_feat["rsi_14"].iloc[-1]) if "rsi_14" in df_feat.columns else 50.0
    rule_dampener, rules_applied = apply_learned_rules(
        ticker, rsi, regime, vix_val, yc, composite)
    composite *= rule_dampener

    composite = float(np.clip(composite, 0, 1))

    # ── OPTIONS IV DAMPENER ──────────────────────────────────
    iv_scale     = 1.0
    iv_note      = ""
    iv_flag_str  = "NORMAL"
    if iv_flag and iv_flag.get("ok"):
        iv_scale    = float(iv_flag.get("position_scale", 1.0))
        iv_flag_str = iv_flag.get("iv_flag", "NORMAL")
        iv_note     = iv_flag.get("note", "")
        if iv_flag_str in ("HIGH", "ELEVATED"):
            # Dampen composite toward 0.5 (uncertainty)
            composite = 0.5 + (composite - 0.5) * iv_scale
            composite = float(np.clip(composite, 0, 1))

    if   composite >= MIN_CONFIDENCE:       action = "BUY"
    elif composite <= (1-MIN_CONFIDENCE):   action = "SELL"
    else:                                   action = "HOLD"

    close = float(df_feat["Close"].iloc[-1])
    atr   = float(df_feat["atr_14"].iloc[-1]) if "atr_14" in df_feat.columns else close*0.02

    return dict(
        ticker=ticker, action=action,
        confidence=round(composite,4),
        p_xgb=round(p_xgb,4), p_lgb=round(p_lgb,4), p_cat=round(p_cat,4),
        p_ensemble=round(p_ens,4),
        garch_p_up=round(garch["p_up"],4),
        ann_vol=round(garch["annvol"],4),
        var95=round(garch["var95"],4),
        sentiment=round(sent,4),
        regime=regime, auc=round(model_pack["auc"],4),
        rsi=round(rsi,2), atr=round(atr,4),
        close=round(close,4),
        rules_applied=rules_applied,
        ts=datetime.datetime.utcnow().isoformat(),
        iv_flag=iv_flag_str,
        iv_scale=round(iv_scale,3),
        iv_note=iv_note
    )

print("Generating signals...")
signals = {}
for tk in models:
    if tk not in featured: continue
    try:
        sig=generate_signal(
            tk,models[tk],featured[tk],
            int(regimes[tk].iloc[-1]) if tk in regimes else 0,
            garch_res.get(tk,dict(p_up=0.5,annvol=0.3,var95=-0.05,es95=-0.08,ok=False)),
            sentiments.get(tk,0.0),
            iv_flag=iv_flags.get(tk, {}))
        signals[tk]=sig
        rules_note = f" [{', '.join(sig['rules_applied'])}]" if sig['rules_applied'] else ""
        print(f"  {sig['action']:<4} {tk:<10} conf={sig['confidence']:.3f}{rules_note}")
    except Exception as ex:
        print(f"  FAIL {tk}: {ex}")
print(f"\n{len(signals)} signals generated")


In [ ]:
# ============================================================
# CELL 12 — CVaR PORTFOLIO OPTIMISATION
# ============================================================
def cvar_optimize(tickers, lookback=252):
    pd_dict={tk:featured[tk]["Close"].tail(lookback)
             for tk in tickers if tk in featured}
    if len(pd_dict)<2:
        n=max(len(tickers),1); return {tk:1/n for tk in tickers}
    prices=pd.DataFrame(pd_dict).dropna()
    rets=prices.pct_change().dropna().values
    T,N=rets.shape
    w=cp.Variable(N,nonneg=True); z=cp.Variable(T,nonneg=True); zeta=cp.Variable()
    prob=cp.Problem(cp.Minimize(zeta+(1/(0.05*T))*cp.sum(z)),
                    [cp.sum(w)==1,w<=0.25,z>=-rets@w-zeta])
    try:
        prob.solve(solver=cp.CLARABEL,verbose=False)
        if w.value is None: raise ValueError("no solution")
        return {tk:float(wt) for tk,wt in zip(pd_dict.keys(),w.value)}
    except Exception as e:
        print(f"  CVaR fallback ({e})")
        n=max(len(tickers),1); return {tk:1/n for tk in tickers}

buy_tickers=[tk for tk,s in signals.items() if s["action"]=="BUY"]
if buy_tickers:
    opt_weights=cvar_optimize(buy_tickers)
    print("CVaR weights:")
    for tk,wt in sorted(opt_weights.items(),key=lambda x:-x[1]):
        print(f"  {tk:<10} {wt:.1%}")
else:
    opt_weights={}
    print("No BUY signals")
print("\nOptimisation complete")


In [ ]:
# ============================================================
# CELL 13 — PAPER TRADE ENGINE + 60-DAY P&L TRACKER
# ============================================================
# Executes paper trades from signals, tracks every position
# mark-to-market, computes realised and unrealised P&L,
# and feeds outcome data into the self-learning loop.
# All data persists to Google Drive.
# ============================================================
import datetime, json
import numpy as np
import pandas as pd
from pathlib import Path

# ── Position sizing ───────────────────────────────────────────
def kelly_qty(confidence, capital, price,
              f=0.25, max_pct=MAX_POSITION_PCT):
    """Half-Kelly position sizing."""
    if price <= 0 or capital <= 0: return 0
    edge  = confidence - (1 - confidence)
    frac  = max(0, edge * f)
    frac  = min(frac, max_pct)
    dollars = capital * frac
    qty   = int(dollars / price)
    return max(qty, 0)

# ── Alpaca integration (optional) ────────────────────────────
def _try_alpaca(action, ticker, qty):
    if not (ALPACA_API_KEY and ALPACA_SECRET_KEY): return None
    try:
        from alpaca.trading.client import TradingClient
        from alpaca.trading.requests import MarketOrderRequest
        from alpaca.trading.enums import OrderSide, TimeInForce
        client = TradingClient(ALPACA_API_KEY, ALPACA_SECRET_KEY, paper=True)
        side   = OrderSide.BUY if action=="BUY" else OrderSide.SELL
        req    = MarketOrderRequest(symbol=ticker, qty=qty,
                                    side=side, time_in_force=TimeInForce.DAY)
        order  = client.submit_order(req)
        return str(order.id)
    except Exception as e:
        return f"alpaca_error:{str(e)[:40]}"

# ── Portfolio equity ──────────────────────────────────────────
def _current_equity():
    """Cash-flow based equity from PT log."""
    try:
        log = pd.read_csv(PT_LOG_FILE)
        bought = (log[log["action"]=="BUY"]["price"]
                  .mul(log[log["action"]=="BUY"]["qty"])).sum()
        sold   = (log[log["action"]=="SELL"]["price"]
                  .mul(log[log["action"]=="SELL"]["qty"])).sum()
        return PORTFOLIO_CAPITAL - bought + sold
    except Exception:
        return PORTFOLIO_CAPITAL

# ── Mark-to-market open positions ────────────────────────────
def get_open_positions():
    """
    Returns DataFrame of open positions with current P&L.
    An open position = a BUY that hasn't been fully offset by a SELL.
    """
    try:
        log = pd.read_csv(PT_LOG_FILE)
        log["qty"] = pd.to_numeric(log["qty"], errors="coerce").fillna(0)
        log["price"] = pd.to_numeric(log["price"], errors="coerce").fillna(0)

        positions = {}
        for _, row in log.iterrows():
            tk  = row["ticker"]
            qty = int(row["qty"])
            if row["action"] == "BUY":
                if tk not in positions:
                    positions[tk] = {"qty":0,"cost":0.0,"entries":[]}
                positions[tk]["qty"]  += qty
                positions[tk]["cost"] += qty * row["price"]
                positions[tk]["entries"].append({"qty":qty,"price":row["price"],"ts":row["ts"]})
            elif row["action"] == "SELL":
                if tk in positions:
                    positions[tk]["qty"]  = max(0, positions[tk]["qty"] - qty)
                    positions[tk]["cost"] = max(0, positions[tk]["cost"] - qty * row["price"])

        # Get current prices for open positions
        rows = []
        for tk, pos in positions.items():
            if pos["qty"] <= 0: continue
            try:
                hist = yf.Ticker(tk).history(period="1d", auto_adjust=True)
                curr_price = float(hist["Close"].iloc[-1]) if not hist.empty else 0.0
            except Exception:
                curr_price = 0.0
            avg_cost = pos["cost"] / pos["qty"] if pos["qty"] > 0 else 0
            mkt_val  = pos["qty"] * curr_price
            unreal_pl= mkt_val - pos["cost"]
            unreal_pct = unreal_pl / pos["cost"] * 100 if pos["cost"] > 0 else 0
            rows.append({
                "ticker":    tk,
                "qty":       pos["qty"],
                "avg_cost":  round(avg_cost, 4),
                "curr_price":round(curr_price, 4),
                "mkt_value": round(mkt_val, 2),
                "cost_basis":round(pos["cost"], 2),
                "unrealised_pl": round(unreal_pl, 2),
                "unrealised_pct":round(unreal_pct, 2),
            })
        return pd.DataFrame(rows) if rows else pd.DataFrame(
            columns=["ticker","qty","avg_cost","curr_price","mkt_value",
                     "cost_basis","unrealised_pl","unrealised_pct"])
    except Exception as e:
        print(f"  open_positions error: {e}")
        return pd.DataFrame()

# ── 60-day P&L summary ───────────────────────────────────────
def compute_60d_pnl():
    """
    Full 60-day P&L breakdown:
    - Realised P&L from closed trades
    - Unrealised P&L from open positions
    - Daily equity curve
    - Per-ticker performance
    - Win rate, Sharpe proxy, max drawdown
    """
    try:
        log = pd.read_csv(PT_LOG_FILE)
        log["ts"]    = pd.to_datetime(log["ts"], errors="coerce")
        log["price"] = pd.to_numeric(log["price"], errors="coerce").fillna(0)
        log["qty"]   = pd.to_numeric(log["qty"],   errors="coerce").fillna(0)

        cutoff = pd.Timestamp.now() - pd.Timedelta(days=60)
        log60  = log[log["ts"] >= cutoff].copy()

        if log60.empty:
            return dict(realised_pl=0, unrealised_pl=0, total_pl=0,
                        win_rate=None, trades_60d=0, equity_curve=[],
                        per_ticker={}, max_drawdown=0, sharpe=None)

        # Realised P&L: match BUYs to SELLs FIFO per ticker
        realised = 0.0
        realised_by_tk = {}
        queues   = {}   # ticker -> deque of (qty, cost)
        from collections import deque
        for _, row in log60.sort_values("ts").iterrows():
            tk  = row["ticker"]
            qty = int(row["qty"])
            px  = float(row["price"])
            if row["action"] == "BUY":
                if tk not in queues: queues[tk] = deque()
                queues[tk].append([qty, px])
            elif row["action"] == "SELL" and tk in queues:
                sell_qty = qty
                while sell_qty > 0 and queues[tk]:
                    entry_qty, entry_px = queues[tk][0]
                    matched = min(sell_qty, entry_qty)
                    pl = matched * (px - entry_px)
                    realised += pl
                    realised_by_tk[tk] = realised_by_tk.get(tk, 0) + pl
                    sell_qty   -= matched
                    queues[tk][0][0] -= matched
                    if queues[tk][0][0] == 0:
                        queues[tk].popleft()

        # Unrealised P&L from open positions
        open_pos = get_open_positions()
        unrealised = float(open_pos["unrealised_pl"].sum()) if not open_pos.empty else 0.0

        # Daily equity curve (simplified)
        daily = log60.copy()
        daily["signed_cash"] = daily.apply(
            lambda r: -r["price"]*r["qty"] if r["action"]=="BUY"
                      else r["price"]*r["qty"] if r["action"]=="SELL" else 0, axis=1)
        daily["date"] = daily["ts"].dt.date
        daily_cash = daily.groupby("date")["signed_cash"].sum().cumsum()
        equity_curve = [(str(d), round(PORTFOLIO_CAPITAL + v, 2))
                        for d, v in daily_cash.items()]

        # Max drawdown
        if equity_curve:
            vals = [v for _,v in equity_curve]
            peak = vals[0]
            mdd  = 0.0
            for v in vals:
                peak = max(peak, v)
                dd   = (peak - v) / peak * 100 if peak > 0 else 0
                mdd  = max(mdd, dd)
        else:
            mdd = 0.0

        # Sharpe proxy (daily returns)
        if len(equity_curve) > 5:
            eq_vals = [v for _,v in equity_curve]
            daily_ret = np.diff(eq_vals) / np.array(eq_vals[:-1])
            sharpe = (np.mean(daily_ret) / np.std(daily_ret) * np.sqrt(252)
                      if np.std(daily_ret) > 0 else None)
        else:
            sharpe = None

        # Win rate from prediction log
        try:
            pred = pd.read_csv(PRED_LOG_FILE)
            pred["ts"] = pd.to_datetime(pred.get("pred_ts", pred.get("ts","")), errors="coerce")
            pred60 = pred[pred["ts"] >= cutoff]
            scored = pred60[pred60["scored"].astype(str)=="True"]
            scored["was_correct"] = scored["was_correct"].astype(str).map(
                {"True":True,"False":False,"true":True,"false":False}).fillna(False)
            win_rate = float(scored["was_correct"].mean()) if len(scored)>0 else None
            n_trades = len(scored)
        except Exception:
            win_rate = None
            n_trades = len(log60)

        total_pl = realised + unrealised

        return dict(
            realised_pl=round(realised, 2),
            unrealised_pl=round(unrealised, 2),
            total_pl=round(total_pl, 2),
            win_rate=win_rate,
            trades_60d=n_trades,
            equity_curve=equity_curve,
            per_ticker=realised_by_tk,
            max_drawdown=round(mdd, 2),
            sharpe=round(sharpe, 3) if sharpe else None,
        )

    except Exception as e:
        print(f"  60d P&L error: {e}")
        return dict(realised_pl=0, unrealised_pl=0, total_pl=0,
                    win_rate=None, trades_60d=0, equity_curve=[],
                    per_ticker={}, max_drawdown=0, sharpe=None)

# ── Prediction logger ─────────────────────────────────────────
def log_prediction(sig):
    """Log every signal to prediction log for outcome scoring."""
    row = {col: None for col in PRED_LOG_COLS}
    row.update({
        "pred_ts":      datetime.datetime.utcnow().isoformat(),
        "ticker":       sig["ticker"],
        "action":       sig["action"],
        "confidence":   sig["confidence"],
        "price_at_pred":sig["close"],
        "p_ensemble":   sig.get("p_ensemble", None),
        "p_up_garch":   sig.get("garch_p_up", None),
        "rsi":          sig["rsi"],
        "regime":       sig["regime"],
        "vix":          MACRO.get("vix"),
        "yield_curve":  MACRO.get("yield_curve"),
        "ism_pmi":      MACRO.get("ism_pmi"),
        "unemployment": MACRO.get("unemployment"),
        "sentiment":    sig.get("sentiment", 0),
        "horizon_days": FORECAST_DAYS,
        "iv_flag":      sig.get("iv_flag", "NORMAL"),
        "iv_scale":     sig.get("iv_scale", 1.0),
        "iv_note":      sig.get("iv_note", ""),
        "scored":       False,
        "outcome_ts":   None,
        "price_at_outcome": None,
        "actual_return":    None,
        "was_correct":      None,
        "magnitude_error":  None,
    })
    try:
        existing = pd.read_csv(PRED_LOG_FILE) if Path(PRED_LOG_FILE).exists() else pd.DataFrame()
        updated  = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
        updated.to_csv(PRED_LOG_FILE, index=False)
    except Exception as e:
        print(f"  log_prediction error: {e}")

# ── Trade executor ────────────────────────────────────────────
def execute_trade(sig, qty, capital):
    """Execute a paper trade and append to PT log."""
    ts     = datetime.datetime.utcnow().isoformat()
    ticker = sig["ticker"]
    action = sig["action"]
    price  = sig["close"]

    if action not in ("BUY","SELL") or qty <= 0:
        return {"status":"skip","reason":"HOLD or qty=0"}

    order_id = _try_alpaca(action, ticker, qty) or "paper"

    row = {
        "ts":        ts,
        "ticker":    ticker,
        "action":    action,
        "price":     round(price, 4),
        "qty":       qty,
        "confidence":round(sig["confidence"], 4),
        "regime":    sig.get("regime", 0),
        "order_id":  order_id,
        "status":    "filled",
        "run_date":  datetime.date.today().isoformat(),
        "iv_flag":   sig.get("iv_flag","NORMAL"),
        "iv_scale":  sig.get("iv_scale",1.0),
        "notional":  round(price * qty, 2),
    }

    try:
        existing = pd.read_csv(PT_LOG_FILE) if Path(PT_LOG_FILE).exists() else pd.DataFrame()
        updated  = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
        updated.to_csv(PT_LOG_FILE, index=False)
    except Exception as e:
        print(f"  execute_trade error: {e}")
        return {"status":"error","reason":str(e)}

    return row

# ── MAIN EXECUTION BLOCK ──────────────────────────────────────
print("\n" + "="*55)
print(" PAPER TRADE ENGINE — v21")
print("="*55)
print(f" Run date: {datetime.date.today()} | Capital: ${PORTFOLIO_CAPITAL:,.0f}")
print(f" Drive: {'mounted' if _drive_mounted else 'session-only'}")

equity = _current_equity()
print(f" Current equity: ${equity:,.2f}")

halt = equity < PORTFOLIO_CAPITAL * (1 - MAX_DRAWDOWN_PCT)
if halt:
    print(f" ⛔ MAX DRAWDOWN HIT — no new trades today")

trade_count = 0
for tk, sig in signals.items():
    log_prediction(sig)
    if halt or sig["action"] == "HOLD":
        continue
    _iv_sc = float(sig.get("iv_scale", 1.0))
    qty    = int(kelly_qty(sig["confidence"], equity, sig["close"]) * _iv_sc)
    if qty == 0:
        continue
    result = execute_trade(sig, qty, equity)
    iv_tag = f" [{sig.get('iv_flag','')} {_iv_sc:.2f}x]" if sig.get("iv_flag","NORMAL")!="NORMAL" else ""
    print(f"  {sig['action']:<4} {tk:<10} qty={qty:3d} @ ${sig['close']:>9,.2f}{iv_tag}")
    trade_count += 1

print(f"\n  {trade_count} trades executed | {len(signals)} predictions logged")

# ── 60-day P&L summary ───────────────────────────────────────
print("\n" + "-"*55)
print(" 60-DAY P&L SUMMARY")
print("-"*55)
pnl = compute_60d_pnl()
print(f"  Realised P&L:    ${pnl['realised_pl']:>+10,.2f}")
print(f"  Unrealised P&L:  ${pnl['unrealised_pl']:>+10,.2f}")
print(f"  Total P&L:       ${pnl['total_pl']:>+10,.2f}  ({pnl['total_pl']/PORTFOLIO_CAPITAL*100:+.1f}%)")
print(f"  Max drawdown:    {pnl['max_drawdown']:.1f}%")
if pnl["win_rate"] is not None:
    print(f"  Win rate:        {pnl['win_rate']:.1%}  ({pnl['trades_60d']} scored predictions)")
if pnl["sharpe"] is not None:
    print(f"  Sharpe proxy:    {pnl['sharpe']:.2f}")

# Open positions
open_pos = get_open_positions()
if not open_pos.empty:
    print(f"\n  Open positions ({len(open_pos)}):")
    for _, row in open_pos.iterrows():
        pl_c = "▲" if row["unrealised_pl"] >= 0 else "▼"
        print(f"    {row['ticker']:<10} {int(row['qty'])} sh @ ${row['avg_cost']:,.2f} → ${row['curr_price']:,.2f} "
              f"{pl_c} ${row['unrealised_pl']:+,.2f} ({row['unrealised_pct']:+.1f}%)")
else:
    print("\n  No open positions yet.")

print("="*55)


In [ ]:
# ============================================================
# CELL 14 — OUTCOME SCORER
# ============================================================
# Checks all unscored predictions where horizon has passed.
# Downloads actual prices, scores each prediction, marks done.

def score_outcomes():
    """Score all mature unscored predictions against actual prices."""
    plog = pd.read_csv(PRED_LOG_FILE)
    if plog.empty:
        print("  No predictions to score yet")
        return pd.DataFrame()

    plog["pred_ts"] = pd.to_datetime(plog["pred_ts"], errors="coerce", utc=True)
    now_utc = pd.Timestamp.utcnow()
    cutoff  = now_utc - pd.Timedelta(days=FORECAST_DAYS+1)

    unscored = plog[(plog["scored"].astype(str)=="False") &
                    (plog["pred_ts"] < cutoff)].copy()

    if unscored.empty:
        print(f"  No mature unscored predictions (need {FORECAST_DAYS}+ days old)")
        return pd.DataFrame()

    print(f"  Scoring {len(unscored)} mature predictions...")
    newly_scored = []

    for idx, row in unscored.iterrows():
        tk = row["ticker"]
        try:
            # Fetch recent prices to find outcome price
            df_now = download_ticker(tk, TRAIN_START, TRAIN_END)
            if df_now is None or df_now.empty:
                continue

            price_at_pred = float(row["price_at_pred"])
            price_now     = float(df_now["Close"].iloc[-1])
            actual_return = (price_now - price_at_pred) / price_at_pred

            # Was the signal directionally correct?
            action = str(row["action"])
            if action == "BUY":
                was_correct = actual_return > 0
            elif action == "SELL":
                was_correct = actual_return < 0
            else:
                was_correct = True  # HOLD always neutral

            conf = float(row["confidence"]) if pd.notna(row["confidence"]) else 0.5
            magnitude_error = abs(actual_return - (conf - 0.5))

            # Update the row
            plog.at[idx, "outcome_ts"]      = now_utc.isoformat()
            plog.at[idx, "price_at_outcome"] = price_now
            plog.at[idx, "actual_return"]    = round(actual_return, 4)
            plog.at[idx, "was_correct"]      = was_correct
            plog.at[idx, "magnitude_error"]  = round(magnitude_error, 4)
            plog.at[idx, "scored"]           = True

            newly_scored.append({
                "ticker":       tk,
                "action":       action,
                "confidence":   conf,
                "was_correct":  was_correct,
                "actual_return":actual_return,
                "regime":       row.get("regime"),
                "vix":          row.get("vix"),
                "rsi":          row.get("rsi"),
                "yield_curve":  row.get("yield_curve"),
                "ism_pmi":      row.get("ism_pmi"),
                "sentiment":    row.get("sentiment"),
            })

            result = "CORRECT" if was_correct else "WRONG"
            print(f"    {result} {tk:<8} {action:<4} "
                  f"pred_px=${price_at_pred:.2f} "
                  f"now=${price_now:.2f} "
                  f"ret={actual_return:+.1%}")

        except Exception as e:
            print(f"    ERROR scoring {tk}: {e}")

    # Save updated prediction log
    plog.to_csv(PRED_LOG_FILE, index=False)
    print(f"  Saved {len(newly_scored)} scored outcomes")
    return pd.DataFrame(newly_scored)

newly_scored_df = score_outcomes()


In [ ]:
# ============================================================
# CELL 15 — FAILURE DIAGNOSIS ENGINE + RULE WRITER
# ============================================================
# Reads all scored outcomes. Finds statistically meaningful
# failure patterns. Writes new rules to LEARNED_RULES.
# River updates ADAPTIVE_WEIGHTS based on recent accuracy.

def diagnose_failures_and_rewrite_rules():
    """
    The self-learning core. Reads scored outcomes, finds where
    the model is systematically wrong, writes rules to fix it.
    """
    global ADAPTIVE_WEIGHTS, LEARNED_RULES

    plog = pd.read_csv(PRED_LOG_FILE)
    scored = plog[plog["scored"].astype(str)=="True"].copy()

    if len(scored) < 5:
        print("  Not enough scored outcomes yet for diagnosis")
        print(f"  Have {len(scored)} — need at least 5")
        return {}, []

    scored["was_correct"] = scored["was_correct"].astype(str).map(
        {"True":True,"False":False,"true":True,"false":False}).fillna(False)
    scored["confidence"]  = pd.to_numeric(scored["confidence"],  errors="coerce").fillna(0.5)
    scored["rsi"]         = pd.to_numeric(scored["rsi"],         errors="coerce").fillna(50)
    scored["vix"]         = pd.to_numeric(scored["vix"],         errors="coerce").fillna(20)
    scored["regime"]      = pd.to_numeric(scored["regime"],      errors="coerce").fillna(1)
    scored["yield_curve"] = pd.to_numeric(scored["yield_curve"], errors="coerce").fillna(0)
    scored["actual_return"]= pd.to_numeric(scored["actual_return"],errors="coerce").fillna(0)

    insights    = []
    new_rules   = dict(LEARNED_RULES)  # start from existing
    MIN_SAMPLES = 3  # need at least this many failures before writing a rule

    overall_acc = scored["was_correct"].mean()
    print(f"  Overall accuracy: {overall_acc:.1%} across {len(scored)} predictions")

    # ── Pattern 1: High RSI + Bear regime ─────────────────────
    mask = (scored["rsi"]>68) & (scored["regime"]==0) & (scored["action"]=="BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = subset["was_correct"].mean()
        if acc < 0.40:  # wrong more than 60% of the time
            dampen = round(min(0.25, (0.50 - acc)), 2)
            new_rules["high_rsi_bear"] = {
                "dampen": dampen, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY with RSI>68 in Bear regime — {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"RULE WRITTEN: High RSI Bear — BUY signals wrong {1-acc:.0%} of time "
                            f"({len(subset)} samples) → dampening confidence by {dampen:.0%}")

    # ── Pattern 2: VIX spike ──────────────────────────────────
    mask = (scored["vix"]>28) & (scored["action"]=="BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = subset["was_correct"].mean()
        if acc < 0.45:
            dampen = round(min(0.20, (0.50 - acc)), 2)
            new_rules["vix_spike"] = {
                "dampen": dampen, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY when VIX>28 — {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"RULE WRITTEN: VIX spike — BUY signals wrong {1-acc:.0%} of time "
                            f"({len(subset)} samples) → dampening by {dampen:.0%}")

    # ── Pattern 3: Inverted yield curve ───────────────────────
    mask = (scored["yield_curve"]<-0.1) & (scored["action"]=="BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = subset["was_correct"].mean()
        if acc < 0.45:
            dampen = round(min(0.20, (0.50 - acc)), 2)
            new_rules["inverted_yc"] = {
                "dampen": dampen, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY with inverted yield curve — {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"RULE WRITTEN: Inverted yield curve — BUY wrong {1-acc:.0%} of time "
                            f"({len(subset)} samples) → dampening by {dampen:.0%}")

    # ── Pattern 4: Per-ticker accuracy ────────────────────────
    for tk in scored["ticker"].unique():
        tk_data = scored[(scored["ticker"]==tk) & (scored["action"]!="HOLD")]
        if len(tk_data) >= 5:
            acc = tk_data["was_correct"].mean()
            if acc < 0.35:
                dampen = round(min(0.20, (0.45 - acc)), 2)
                rkey = f"ticker_{tk}"
                new_rules[rkey] = {
                    "dampen": dampen, "count": int(len(tk_data)),
                    "accuracy": round(acc, 3),
                    "description": f"{tk} systematically underperforming — {acc:.0%} accurate ({len(tk_data)} samples)"
                }
                insights.append(f"RULE WRITTEN: {tk} — only {acc:.0%} accurate "
                                f"({len(tk_data)} samples) → dampening by {dampen:.0%}")

    # ── Pattern 5: Remove stale rules (if model improved) ─────
    for key in list(new_rules.keys()):
        rule = new_rules[key]
        if rule.get("accuracy", 0) > 0.55 and rule.get("count", 0) >= 10:
            del new_rules[key]
            insights.append(f"RULE REMOVED: {key} — model has improved to {rule['accuracy']:.0%} accuracy")

    # ── River adaptive weight update ──────────────────────────
    recent = scored.tail(30)  # last 30 scored predictions
    if len(recent) >= 5:
        # Calculate feature-level accuracy
        ens_corr  = recent[recent["was_correct"]==True]["confidence"].mean()
        ens_wrong = recent[recent["was_correct"]==False]["confidence"].mean()

        # If ensemble is overconfident on wrong predictions, reduce weight
        if pd.notna(ens_wrong) and pd.notna(ens_corr):
            if ens_wrong > 0.63:  # overconfident on wrong predictions
                new_w_ens = max(0.40, ADAPTIVE_WEIGHTS["w_ensemble"] - 0.02)
                new_w_garch = min(0.28, ADAPTIVE_WEIGHTS["w_garch"] + 0.01)
                new_w_sent  = min(0.18, ADAPTIVE_WEIGHTS["w_sentiment"] + 0.01)
                ADAPTIVE_WEIGHTS["w_ensemble"]  = round(new_w_ens, 3)
                ADAPTIVE_WEIGHTS["w_garch"]     = round(new_w_garch, 3)
                ADAPTIVE_WEIGHTS["w_sentiment"] = round(new_w_sent, 3)
                insights.append(f"WEIGHT UPDATE: Ensemble overconfident on wrong predictions "
                                f"(avg conf={ens_wrong:.3f}). "
                                f"Reduced w_ensemble to {ADAPTIVE_WEIGHTS['w_ensemble']:.3f}")
            elif ens_wrong < 0.58 and ens_corr > 0.70:
                # Ensemble is well-calibrated — boost its weight
                new_w_ens = min(0.65, ADAPTIVE_WEIGHTS["w_ensemble"] + 0.01)
                ADAPTIVE_WEIGHTS["w_ensemble"] = round(new_w_ens, 3)
                insights.append(f"WEIGHT UPDATE: Ensemble well-calibrated — "
                                f"increased w_ensemble to {ADAPTIVE_WEIGHTS['w_ensemble']:.3f}")

    # Renormalise weights to sum to 1.0
    total = sum(ADAPTIVE_WEIGHTS.values())
    for k in ADAPTIVE_WEIGHTS:
        ADAPTIVE_WEIGHTS[k] = round(ADAPTIVE_WEIGHTS[k]/total, 4)

    # Save everything to Drive
    LEARNED_RULES = new_rules
    Path(WEIGHTS_FILE).write_text(json.dumps(ADAPTIVE_WEIGHTS, indent=2))
    Path(RULES_FILE).write_text(json.dumps(LEARNED_RULES, indent=2))

    print(f"  Adaptive weights: {ADAPTIVE_WEIGHTS}")
    print(f"  Active learned rules: {len(LEARNED_RULES)}")
    return new_rules, insights

print("Running failure diagnosis and rule-writing engine...")
new_rules, insights = diagnose_failures_and_rewrite_rules()

if insights:
    print("\nInsights:")
    for ins in insights:
        print(f"  > {ins}")
else:
    print("\nNo new rules written — accumulating more data...")


In [ ]:
# ============================================================
# CELL 16 — QUANT TERMINAL v21 HOMEPAGE
# ============================================================
# Renders automatically on Run All.
# Search: change SEARCH_TICKER and press Shift+Enter.
# ============================================================
%matplotlib inline
from IPython.display import display, HTML, clear_output
import datetime

# =============================================
SEARCH_TICKER = "NONE"
# Change to any ticker to run deep analysis:
# e.g.  AAPL  NVDA  TSLA  BTC-USD  GOOGL
# Set to "NONE" to show dashboard only
# =============================================

def _fmt(val, fmt=".2f", suffix="", prefix=""):
    if val is None: return "N/A"
    try: return f"{prefix}{val:{fmt}}{suffix}"
    except Exception: return str(val)

def _bar(pct, color):
    w = max(2, min(int(pct), 100))
    return (f'<div style="height:3px;background:var(--color-background-secondary);'
            f'border-radius:2px;margin-top:4px">'
            f'<div style="width:{w}%;height:3px;background:{color};border-radius:2px"></div></div>')

def _macro_card(label, val_str, sub, bar_pct, bar_color, sub_color=None):
    sc = sub_color or "var(--color-text-secondary)"
    return (
        f'<div style="background:var(--color-background-primary);border:0.5px solid '
        f'var(--color-border-tertiary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);margin-bottom:3px;'
        f'letter-spacing:.04em">{label}</div>'
        f'<div style="font-size:16px;font-weight:500;color:var(--color-text-primary);'
        f'margin-bottom:1px">{val_str}</div>'
        f'<div style="font-size:10px;color:{sc}">{sub}</div>'
        f'{_bar(bar_pct, bar_color)}</div>'
    )


def generate_reasoning(sig, garch, sent_sc, regime, regime_labels, mp):
    action=sig["action"]; conf=sig["confidence"]; rsi=sig["rsi"]
    p_ens=sig["p_ensemble"]; p_up=garch["p_up"]; ann_vol=garch["annvol"]
    vix=MACRO.get("vix") or 20; yc=MACRO.get("yield_curve") or 0
    pmi=MACRO.get("ism_pmi") or 50; auc=mp["auc"]
    reasons=[]; cautions=[]
    if p_ens>=0.70: reasons.append(f"ML ensemble strongly bullish ({p_ens:.0%} upside probability across all three models).")
    elif p_ens>=0.60: reasons.append(f"ML ensemble moderately bullish ({p_ens:.0%} upside probability).")
    elif p_ens<=0.35: cautions.append(f"ML ensemble bearish ({p_ens:.0%} upside probability).")
    elif p_ens<=0.45: cautions.append(f"ML ensemble leaning bearish ({p_ens:.0%}).")
    else: reasons.append(f"ML ensemble neutral ({p_ens:.0%}) — no strong directional edge.")
    if p_up>=0.60: reasons.append(f"GARCH Monte Carlo shows {p_up:.0%} probability of positive returns over {FORECAST_DAYS} days.")
    elif p_up<=0.40: cautions.append(f"GARCH gives only {p_up:.0%} upside probability.")
    if ann_vol>0.50: cautions.append(f"Elevated annualized volatility at {ann_vol:.0%}.")
    if rsi>70: cautions.append(f"RSI overbought at {rsi:.1f} — short-term pullback risk.")
    elif rsi<30: reasons.append(f"RSI oversold at {rsi:.1f} — potential mean-reversion.")
    elif 45<=rsi<=60: reasons.append(f"RSI healthy at {rsi:.1f}.")
    if sent_sc>0.15: reasons.append(f"News sentiment positive ({sent_sc:+.2f}).")
    elif sent_sc<-0.15: cautions.append(f"News sentiment negative ({sent_sc:+.2f}).")
    if regime==1: reasons.append(f"Stock in Bull/Trending regime (HMM state 1).")
    elif regime==0: cautions.append(f"Stock in Bear/Volatile regime (HMM state 0).")
    macro_notes=[]
    if vix>25: cautions.append(f"VIX elevated at {vix:.1f} — confidence dampened.")
    elif vix<16: macro_notes.append(f"low VIX ({vix:.1f})")
    if yc>0: macro_notes.append(f"normal yield curve (+{yc:.2f}%)")
    elif yc<0: cautions.append(f"Inverted yield curve ({yc:+.2f}%).")
    if pmi>50: macro_notes.append(f"ISM PMI {pmi:.1f} (expansion)")
    elif pmi<50: cautions.append(f"ISM PMI below 50 ({pmi:.1f}).")
    if macro_notes: reasons.append(f"Macro supportive: {', '.join(macro_notes)}.")
    if auc>=0.60: reasons.append(f"Model AUC {auc:.3f} — statistically meaningful predictions.")
    elif auc<0.52: cautions.append(f"Low model AUC ({auc:.3f}) — treat with caution.")
    rules=sig.get("rules_applied",[])
    if rules: cautions.append(f"Self-written rules active: {', '.join(rules)}.")
    if action=="BUY": summary=f"Signal: BUY with {conf:.0%} confidence. Bullish factors outweigh bearish ones."
    elif action=="SELL": summary=f"Signal: SELL with {1-conf:.0%} bearish confidence. Downside risk dominates."
    else: summary=f"Signal: HOLD — confidence ({conf:.3f}) between buy ({MIN_CONFIDENCE:.2f}) and sell ({1-MIN_CONFIDENCE:.2f}) thresholds."
    return dict(summary=summary,reasons=reasons,cautions=cautions,action=action)


def on_demand_analysis(ticker):
    ticker=ticker.strip().upper()
    if not ticker or ticker=="NONE": return
    display(HTML(
        f'<div style="background:var(--color-background-secondary);border-radius:8px;'
        f'padding:12px 16px;font-family:monospace;margin:8px 0">'
        f'<span style="font-size:13px;font-weight:500;color:var(--color-text-info)">Analyzing: {ticker}</span>'
        f'<span style="font-size:11px;color:var(--color-text-secondary);margin-left:10px">running full pipeline...</span>'
        f'</div>'))
    df_raw=download_ticker(ticker,TRAIN_START,TRAIN_END)
    if df_raw is None or len(df_raw)<200:
        display(HTML(f'<div style="color:var(--color-text-danger);padding:8px">Could not download data for {ticker}</div>'))
        return
    print(f"  Data: {len(df_raw)} rows | ${df_raw['Close'].iloc[-1]:.2f}")
    try: df_feat=build_features(df_raw)
    except Exception as e: print(f"  Feature error: {e}"); return
    if ticker in models:
        mp=models[ticker]; print(f"  Model (cached) AUC={mp['auc']:.3f}")
    else:
        print(f"  Training model for {ticker}...")
        try:
            mp=train_ensemble(df_feat,ticker); models[ticker]=mp; featured[ticker]=df_feat
            print(f"  Trained AUC={mp['auc']:.3f}")
        except Exception as e: print(f"  Training failed: {e}"); return
    reg_series=fit_hmm(df_raw); regime=int(reg_series.iloc[-1])
    regime_labels={0:"Bear/Volatile",1:"Bull/Trending",2:"Neutral/Sideways"}
    garch=garch_vol_forecast(df_feat,ticker,n_paths=1000)
    headlines=fetch_headlines(ticker,n=6); sent_sc=sentiment_score(headlines)
    sig=generate_signal(ticker,mp,df_feat,regime,garch,sent_sc)
    equity_now=_current_equity()
    qty=kelly_qty(sig["confidence"],equity_now,sig["close"])
    dollar_exp=qty*sig["close"]
    aclr={"BUY":"var(--color-text-success)","SELL":"var(--color-text-danger)","HOLD":"var(--color-text-warning)"}.get(sig["action"],"var(--color-text-primary)")
    abg={"BUY":"var(--color-background-success)","SELL":"var(--color-background-danger)","HOLD":"var(--color-background-warning)"}.get(sig["action"],"var(--color-background-secondary)")
    display(HTML(
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:10px;overflow:hidden;margin:8px 0;font-family:var(--font-sans)">'
        f'<div style="background:var(--color-background-secondary);padding:10px 16px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;justify-content:space-between;align-items:center">'
        f'<span style="font-size:15px;font-weight:500;color:var(--color-text-primary)">{ticker}</span>'
        f'<span style="background:{abg};color:{aclr};padding:3px 12px;border-radius:6px;font-size:11px;font-weight:500">{sig["action"]}</span></div>'
        f'<div style="display:grid;grid-template-columns:repeat(3,1fr)">'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary);border-bottom:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">CONFIDENCE</div><div style="font-size:18px;font-weight:500;color:{aclr}">{sig["confidence"]:.3f}</div></div>'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary);border-bottom:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">CLOSE PRICE</div><div style="font-size:18px;font-weight:500;color:var(--color-text-primary)">${sig["close"]:,.2f}</div></div>'
        f'<div style="padding:10px 14px;border-bottom:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">POSITION</div><div style="font-size:18px;font-weight:500;color:var(--color-text-info)">{qty} sh = ${dollar_exp:,.0f}</div></div>'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">RSI (14)</div><div style="font-size:16px;font-weight:500;color:{"var(--color-text-danger)" if sig["rsi"]>70 else "var(--color-text-success)" if sig["rsi"]<30 else "var(--color-text-primary)"}">{sig["rsi"]:.1f}</div></div>'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">ANN VOL</div><div style="font-size:16px;font-weight:500;color:var(--color-text-primary)">{garch["annvol"]:.1%}</div></div>'
        f'<div style="padding:10px 14px"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">SENTIMENT</div><div style="font-size:16px;font-weight:500;color:{"var(--color-text-success)" if sent_sc>0 else "var(--color-text-danger)"}">{sent_sc:+.3f}</div></div>'
        f'</div>'
        f'<div style="padding:8px 14px;border-top:0.5px solid var(--color-border-tertiary);font-size:11px;color:var(--color-text-secondary)">'
        f'XGB {sig["p_xgb"]:.3f} | LGB {sig["p_lgb"]:.3f} | CAT {sig["p_cat"]:.3f} | '
        f'Regime: {regime_labels.get(regime,"?")} | GARCH p_up: {garch["p_up"]:.1%} | AUC: {mp["auc"]:.3f}</div></div>'))
    r=generate_reasoning(sig,garch,sent_sc,regime,regime_labels,mp)
    aclr2={"BUY":"var(--color-text-success)","SELL":"var(--color-text-danger)","HOLD":"var(--color-text-warning)"}.get(r["action"],"var(--color-text-primary)")
    li_r='style="margin-bottom:5px;color:var(--color-text-primary)"'
    li_c='style="margin-bottom:5px;color:var(--color-text-warning)"'
    ri="".join(f"<li {li_r}>{pt}</li>" for pt in r["reasons"])
    ci="".join(f"<li {li_c}>{pt}</li>" for pt in r["cautions"])
    wb_bg="background:var(--color-background-warning);border-radius:var(--border-radius-md);border:0.5px solid var(--color-border-warning)"
    cb=(f'<div style="margin-top:10px;padding:10px 12px;{wb_bg}"><div style="font-size:11px;font-weight:500;color:var(--color-text-warning);margin-bottom:6px">Cautions</div><ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{ci}</ul></div>' if ci else "")
    display(HTML(
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:var(--border-radius-lg);padding:14px 16px;margin:8px 0;font-family:var(--font-sans)">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.07em;margin-bottom:8px">Why {ticker} is a {r["action"]}</div>'
        f'<div style="font-size:13px;font-weight:500;color:{aclr2};margin-bottom:10px">{r["summary"]}</div>'
        f'<ul style="font-size:12px;line-height:1.7;margin:0;padding-left:16px;color:var(--color-text-primary)">{ri}</ul>'
        f'{cb}</div>'))
    fig,axes=plt.subplots(3,1,figsize=(16,11),sharex=True,facecolor="#0a0e1a")
    fig.suptitle(f"{ticker}  |  {sig['action']}  |  conf={sig['confidence']:.3f}",
                 fontsize=12,fontweight="bold",color="#e2e8f0")
    plot_df=df_feat.tail(180).copy()
    for ax in axes:
        ax.set_facecolor("#0d1220"); ax.tick_params(colors="#475569",labelsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor("#1e2530")
    ax=axes[0]
    ax.plot(plot_df.index,plot_df["Close"],color="#e2e8f0",lw=1.5,label="Close")
    for w,clr in [(20,"#378ADD"),(50,"#f59e0b"),(200,"#a855f7")]:
        if f"sma_{w}" in plot_df.columns:
            ax.plot(plot_df.index,plot_df[f"sma_{w}"],lw=0.8,alpha=0.7,label=f"SMA{w}",color=clr)
    if "bb_upper" in plot_df.columns:
        ax.fill_between(plot_df.index,plot_df["bb_upper"],plot_df["bb_lower"],alpha=0.06,color="#378ADD")
    ax.set_ylabel("Price",color="#475569",fontsize=9)
    ax.legend(fontsize=8,facecolor="#0d1220",edgecolor="#1e2530",labelcolor="#94a3b8")
    ax=axes[1]
    if "rsi_14" in plot_df.columns:
        ax.plot(plot_df.index,plot_df["rsi_14"],color="#f87171",lw=1)
        ax.axhline(70,color="#ef4444",linestyle="--",alpha=0.4,lw=0.8)
        ax.axhline(30,color="#4ade80",linestyle="--",alpha=0.4,lw=0.8)
        ax.set_ylim(0,100); ax.set_ylabel("RSI(14)",color="#475569",fontsize=9)
        ax.fill_between(plot_df.index,plot_df["rsi_14"],50,where=plot_df["rsi_14"]>50,alpha=0.08,color="#4ade80")
        ax.fill_between(plot_df.index,plot_df["rsi_14"],50,where=plot_df["rsi_14"]<50,alpha=0.08,color="#f87171")
    ax=axes[2]
    if "macd" in plot_df.columns and "macd_sig" in plot_df.columns:
        ax.plot(plot_df.index,plot_df["macd"],color="#7dd3fc",lw=1,label="MACD")
        ax.plot(plot_df.index,plot_df["macd_sig"],color="#f87171",lw=1,label="Signal")
        if "macd_h" in plot_df.columns:
            ax.bar(plot_df.index,plot_df["macd_h"],
                   color=["#4ade80" if v>=0 else "#f87171" for v in plot_df["macd_h"]],alpha=0.5)
        ax.axhline(0,color="#475569",lw=0.5); ax.set_ylabel("MACD",color="#475569",fontsize=9)
        ax.legend(fontsize=8,facecolor="#0d1220",edgecolor="#1e2530",labelcolor="#94a3b8")
    plt.tight_layout()
    plt.savefig(f"search_{ticker.replace('-','_')}_v21.png",dpi=110,bbox_inches="tight",facecolor="#0a0e1a")
    plt.show(); print(f"  Chart saved.")


def render_homepage():
    now   = datetime.datetime.now().strftime("%b %d %Y  %H:%M")
    m     = MACRO
    eq    = _current_equity()
    pnl   = eq - PORTFOLIO_CAPITAL
    pnl_c = "var(--color-text-success)" if pnl >= 0 else "var(--color-text-danger)"

    # accuracy stats from pred log
    try:
        import pandas as _pd
        plog   = _pd.read_csv(PRED_LOG_FILE)
        scored = plog[plog["scored"].astype(str)=="True"].copy()
        scored["was_correct"] = scored["was_correct"].astype(str).map(
            {"True":True,"False":False,"true":True,"false":False}).fillna(False)
        n_scored    = len(scored)
        overall_acc = scored["was_correct"].mean() if n_scored > 0 else None
        recent_acc  = scored.tail(10)["was_correct"].mean() if n_scored >= 3 else None
        n_correct   = int(scored["was_correct"].sum()) if n_scored > 0 else 0
    except Exception:
        n_scored = 0; overall_acc = None; recent_acc = None; n_correct = 0

    def acc_color(a):
        if a is None: return "var(--color-text-secondary)"
        return ("var(--color-text-success)" if a >= 0.60 else
                "var(--color-text-warning)" if a >= 0.50 else
                "var(--color-text-danger)")

    acc_str = f"{overall_acc:.1%}" if overall_acc is not None else "—"
    rec_str = f"{recent_acc:.1%}"  if recent_acc  is not None else "—"

    # macro cards
    vix   = m.get("vix") or 20
    tnx   = m.get("tnx_10y") or 4.3
    yc    = m.get("yield_curve") or 0
    dxy   = m.get("dxy") or 104
    wti   = m.get("wti_crude") or 80
    gold  = m.get("gold") or 2000
    unemp = m.get("unemployment") or 3.9
    cpi   = m.get("cpi_yoy") or 3.1
    gdp   = m.get("gdp_growth") or 2.8
    pmi   = m.get("ism_pmi") or 50.3
    cs    = m.get("credit_spread")
    ey    = m.get("earnings_yield")
    cfg   = m.get("crypto_fg") or 50
    cfg_l = m.get("crypto_fg_label","—")
    rml   = m.get("macro_regime","—")
    ret   = m.get("retail_sales")

    macro_html = "".join([
        _macro_card("Fed funds rate",   _fmt(m.get("fed_rate"),".2f","%"),
            "interest rate env",        min((m.get("fed_rate") or 4)/8*100,100), "#378ADD"),
        _macro_card("10Y Treasury",     _fmt(tnx,".2f","%"),
            "risk-free benchmark",      min(tnx/8*100,100),
            "#E24B4A" if tnx>5 else "#BA7517" if tnx>4 else "#639922"),
        _macro_card("Yield curve",      _fmt(yc,"+.2f","%"),
            "normal" if yc>0 else "inverted — risk",
            50+yc*20,
            "#639922" if yc>0 else "#E24B4A",
            sub_color="#639922" if yc>0 else "#E24B4A"),
        _macro_card("VIX fear index",   _fmt(vix,".1f"),
            "extreme fear" if vix>30 else "elevated" if vix>20 else "low — risk-on",
            min(vix/50*100,100),
            "#E24B4A" if vix>25 else "#BA7517" if vix>18 else "#639922",
            sub_color="#E24B4A" if vix>25 else "#BA7517" if vix>18 else "#639922"),
        _macro_card("Unemployment",     _fmt(unemp,".1f","%"),
            "above 5% — watch" if unemp>5 else "healthy labor market",
            min(unemp/10*100,100),
            "#E24B4A" if unemp>5.5 else "#BA7517" if unemp>4.5 else "#639922"),
        _macro_card("CPI inflation",    _fmt(cpi,".1f","% YoY"),
            "above target" if cpi>2.5 else "near target",
            min(cpi/8*100,100),
            "#E24B4A" if cpi>4 else "#BA7517" if cpi>2.5 else "#639922",
            sub_color="#E24B4A" if cpi>4 else "#BA7517" if cpi>2.5 else "#639922"),
        _macro_card("GDP growth",       _fmt(gdp,"+.1f","% QoQ"),
            "contraction" if gdp<0 else "moderate" if gdp<3 else "strong",
            max(0,min((gdp+2)/8*100,100)),
            "#E24B4A" if gdp<0 else "#BA7517" if gdp<1.5 else "#639922"),
        _macro_card("ISM PMI",          _fmt(pmi,".1f"),
            "contraction" if pmi<50 else "expansion",
            min(pmi/70*100,100),
            "#E24B4A" if pmi<48 else "#BA7517" if pmi<50 else "#639922",
            sub_color="#E24B4A" if pmi<48 else "#BA7517" if pmi<50 else "#639922"),
        _macro_card("WTI crude oil",    _fmt(wti,".1f","  $/bbl"),
            "inflationary" if wti>90 else "moderate",
            min(wti/120*100,100),
            "#E24B4A" if wti>90 else "#BA7517" if wti>75 else "#639922"),
        _macro_card("Gold",             "$"+_fmt(gold,",.0f"),
            "fear / inflation hedge",   min(gold/3500*100,100), "#BA7517"),
        _macro_card("DXY dollar",       _fmt(dxy,".1f"),
            "strong USD" if dxy>105 else "neutral",
            min(dxy/115*100,100),
            "#BA7517" if dxy>105 else "#639922"),
        _macro_card("Credit spread",    _fmt(cs,".4f") if cs else "N/A",
            "HYG/LQD — lower = stress",
            min((cs or 1)*50,100) if cs else 50,
            "#E24B4A" if cs and cs<0.90 else "#639922"),
        _macro_card("Retail sales",     _fmt(ret,"+.1f","% MoM") if ret else "N/A",
            "consumer spending",
            max(0,min(50+(ret or 0)*10,100)),
            "#639922" if (ret or 0)>0 else "#E24B4A"),
        _macro_card("Earnings yield",   _fmt(ey,".2f","%") if ey else "N/A",
            f"vs bonds: {_fmt(m.get('ey_spread'),'+.2f','%') if m.get('ey_spread') else 'N/A'}",
            min((ey or 4)/8*100,100) if ey else 50,
            "#639922" if (m.get("ey_spread") or 0)>0 else "#E24B4A"),
        _macro_card("Crypto fear/greed",str(cfg),
            cfg_l,  cfg,
            "#E24B4A" if cfg<25 else "#BA7517" if cfg<45 else
            "#639922" if cfg<75 else "#E24B4A"),
        _macro_card("Market regime",
            rml.split("/")[0] if "/" in rml else rml,
            f"VIX {_fmt(vix,'.1f')}",
            100-min(vix/50*100,100),
            "#E24B4A" if "Bear" in rml else "#BA7517" if "Neutral" in rml else "#639922"),
    ])

    # signal rows
    sig_rows = ""
    for tk, sig in sorted(signals.items(), key=lambda x:-x[1]["confidence"]):
        a=sig["action"]; conf=sig["confidence"]; rsi=sig["rsi"]
        close=sig["close"]; av=sig["ann_vol"]; sent=sig["sentiment"]
        regime=sig["regime"]; rules=sig.get("rules_applied",[])
        rlbl={0:"Bear",1:"Bull",2:"Neutral"}.get(regime,"?")
        if a=="BUY":
            badge='<span style="background:var(--color-background-success);color:var(--color-text-success);padding:2px 8px;border-radius:6px;font-size:10px;font-weight:500">BUY</span>'
        elif a=="SELL":
            badge='<span style="background:var(--color-background-danger);color:var(--color-text-danger);padding:2px 8px;border-radius:6px;font-size:10px;font-weight:500">SELL</span>'
        else:
            badge='<span style="background:var(--color-background-warning);color:var(--color-text-warning);padding:2px 8px;border-radius:6px;font-size:10px;font-weight:500">HOLD</span>'
        bw=int(conf*72)
        bc=("var(--color-text-success)" if conf>=MIN_CONFIDENCE else
            "var(--color-text-danger)" if conf<=(1-MIN_CONFIDENCE) else
            "var(--color-text-warning)")
        rsi_c=("var(--color-text-danger)" if rsi>70 else
               "var(--color-text-success)" if rsi<30 else
               "var(--color-text-secondary)")
        s_c=("var(--color-text-success)" if sent>0 else
             "var(--color-text-danger)" if sent<0 else
             "var(--color-text-secondary)")
        rbg=("var(--color-background-success)" if rlbl=="Bull" else
             "var(--color-background-danger)" if rlbl=="Bear" else
             "var(--color-background-warning)")
        rc=("var(--color-text-success)" if rlbl=="Bull" else
            "var(--color-text-danger)"  if rlbl=="Bear" else
            "var(--color-text-warning)")
        rule_note=""
        iv_note_html=""
        if rules:
            rule_note=(f'<div style="font-size:9px;color:var(--color-text-warning);'
                       f'margin-top:2px">{" · ".join(rules)}</div>')
        _iv_f = sig.get("iv_flag","NORMAL")
        _iv_n = sig.get("iv_note","")
        if _iv_f in ("HIGH","ELEVATED"):
            iv_note_html=(f'<div style="font-size:9px;color:var(--color-text-danger);margin-top:1px">'
                          f'⚠ {_iv_f} IV · {_iv_n[:40]}</div>')
        sig_rows += (
            f'<tr style="border-bottom:0.5px solid var(--color-border-tertiary)">'
            f'<td style="padding:8px 12px;font-weight:500;font-size:12px;'
            f'color:var(--color-text-primary)">{tk}</td>'
            f'<td style="padding:8px 12px;text-align:center">{badge}</td>'
            f'<td style="padding:8px 12px"><div style="display:flex;align-items:center;gap:5px">'
            f'<div style="width:72px;height:4px;background:var(--color-background-secondary);border-radius:2px">'
            f'<div style="width:{bw}px;height:4px;background:{bc};border-radius:2px"></div></div>'
            f'<span style="font-size:11px;font-weight:500;color:{bc}">{conf:.3f}</span></div>'
            f'{rule_note}{iv_note_html}</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:var(--color-text-secondary);text-align:right">'
            f'${close:,.2f}</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:{rsi_c};text-align:right">{rsi:.1f}</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:{s_c};text-align:right">{sent:+.3f}</td>'
            f'<td style="padding:8px 12px;text-align:center">'
            f'<span style="background:{rbg};color:{rc};padding:1px 6px;border-radius:4px;font-size:10px">'
            f'{rlbl}</span></td>'
            f'</tr>'
        )

    # learned rules
    rules_html = ""
    if LEARNED_RULES:
        for key, rule in LEARNED_RULES.items():
            rules_html += (
                f'<div style="padding:6px 12px;border-bottom:0.5px solid var(--color-border-tertiary)">'
                f'<div style="display:flex;justify-content:space-between;align-items:center">'
                f'<span style="font-size:12px;color:var(--color-text-primary)">{rule.get("description","")}</span>'
                f'<span style="font-size:11px;color:var(--color-text-warning);white-space:nowrap;margin-left:8px">'
                f'dampen {rule.get("dampen",0):.0%}</span></div>'
                f'<div style="font-size:10px;color:var(--color-text-secondary);margin-top:1px">'
                f'{rule.get("count",0)} samples · {rule.get("accuracy",0):.1%} accuracy</div></div>'
            )
    else:
        rules_html = ('<div style="padding:10px 12px;font-size:12px;color:var(--color-text-secondary)">'
                      'No rules yet — accumulating scored outcomes...</div>')

    # recent outcomes
    outcomes_html = ""
    try:
        import pandas as _pd2
        sc2 = _pd2.read_csv(PRED_LOG_FILE)
        sc2 = sc2[sc2["scored"].astype(str)=="True"].copy()
        sc2["was_correct"] = sc2["was_correct"].astype(str).map(
            {"True":True,"False":False,"true":True,"false":False}).fillna(False)
        sc2["actual_return"] = _pd2.to_numeric(sc2["actual_return"], errors="coerce").fillna(0)
        for _,row in sc2.tail(6).iterrows():
            correct=bool(row["was_correct"]); ret2=float(row["actual_return"])
            ok_c="var(--color-text-success)" if correct else "var(--color-text-danger)"
            ok_l="CORRECT" if correct else "WRONG"
            outcomes_html += (
                f'<div style="display:flex;justify-content:space-between;align-items:center;'
                f'padding:4px 0;border-bottom:0.5px solid var(--color-border-tertiary)">'
                f'<span style="font-size:11px;font-weight:500;color:var(--color-text-primary);width:60px">'
                f'{row["ticker"]}</span>'
                f'<span style="font-size:10px;color:var(--color-text-secondary)">{str(row["action"])}</span>'
                f'<span style="font-size:11px;font-weight:500;color:{ok_c}">{ok_l}</span>'
                f'<span style="font-size:11px;color:var(--color-text-secondary)">{ret2:+.1%}</span>'
                f'</div>'
            )
    except Exception:
        pass
    if not outcomes_html:
        outcomes_html = ('<div style="font-size:12px;color:var(--color-text-secondary);padding:8px 0">'
                         f'Predictions score after {FORECAST_DAYS} trading days</div>')

    buy_n  = sum(1 for s in signals.values() if s["action"]=="BUY")
    sell_n = sum(1 for s in signals.values() if s["action"]=="SELL")
    hold_n = sum(1 for s in signals.values() if s["action"]=="HOLD")
    avg_c  = sum(s["confidence"] for s in signals.values()) / max(len(signals),1)

    display(HTML(
        f'<div style="font-family:var(--font-sans)">'

        # ── topbar
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;padding:10px 16px;display:flex;justify-content:space-between;'
        f'align-items:center;margin-bottom:8px">'
        f'<span style="font-size:14px;font-weight:500;color:var(--color-text-primary)">'
        f'Quant Terminal v21</span>'
        f'<span style="font-size:11px;color:var(--color-text-secondary)">{now}</span></div>'

        # ── search bar
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-info);'
        f'border-radius:8px;padding:10px 16px;margin-bottom:8px">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-secondary);margin-bottom:4px">'
        f'SEARCH — change <code style="background:var(--color-background-secondary);'
        f'padding:1px 6px;border-radius:4px;color:var(--color-text-info)">SEARCH_TICKER</code>'
        f' at the top of this cell and press <kbd style="background:var(--color-background-secondary);'
        f'padding:1px 6px;border-radius:4px;font-size:10px">Shift+Enter</kbd>'
        f' for any stock or crypto analysis</div>'
        f'<div style="font-size:12px;color:var(--color-text-secondary)">'
        f'e.g. &nbsp;NVDA &nbsp;TSLA &nbsp;MSFT &nbsp;GOOGL &nbsp;AMZN &nbsp;BTC-USD &nbsp;ETH-USD</div></div>'

        # ── stats row
        f'<div style="display:grid;grid-template-columns:repeat(6,1fr);gap:6px;margin-bottom:8px">'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">BUY</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-success)">{buy_n}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">SELL</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-danger)">{sell_n}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">HOLD</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-warning)">{hold_n}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">Avg conf</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-primary)">{avg_c:.3f}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">Accuracy</div>'
        f'<div style="font-size:20px;font-weight:500;color:{acc_color(overall_acc)}">{acc_str}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">P&amp;L</div>'
        f'<div style="font-size:20px;font-weight:500;color:{pnl_c}">${pnl:+,.0f}</div></div>'
        f'</div>'

        # ── macro section
        f'<div style="font-size:10px;font-weight:500;color:var(--color-text-secondary);'
        f'text-transform:uppercase;letter-spacing:.07em;margin:0 0 6px 2px">Macro environment</div>'
        f'<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:6px;margin-bottom:8px">'
        f'{macro_html}</div>'

        # ── signal table
        f'<div style="font-size:10px;font-weight:500;color:var(--color-text-secondary);'
        f'text-transform:uppercase;letter-spacing:.07em;margin:0 0 5px 2px">Signal dashboard</div>'
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;overflow:hidden;margin-bottom:8px">'
        f'<table style="width:100%;border-collapse:collapse">'
        f'<thead><tr style="background:var(--color-background-secondary);'
        f'border-bottom:0.5px solid var(--color-border-tertiary)">'
        f'<th style="padding:7px 12px;text-align:left;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">TICKER</th>'
        f'<th style="padding:7px 12px;text-align:center;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">SIGNAL</th>'
        f'<th style="padding:7px 12px;text-align:left;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">CONFIDENCE</th>'
        f'<th style="padding:7px 12px;text-align:right;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">PRICE</th>'
        f'<th style="padding:7px 12px;text-align:right;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">RSI</th>'
        f'<th style="padding:7px 12px;text-align:right;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">SENTIMENT</th>'
        f'<th style="padding:7px 12px;text-align:center;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">REGIME</th>'
        f'</tr></thead><tbody>{sig_rows}</tbody></table></div>'

        # ── bottom row: self-learning status + recent outcomes
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px">'

        # self-written rules
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;overflow:hidden">'
        f'<div style="background:var(--color-background-secondary);padding:8px 12px;'
        f'border-bottom:0.5px solid var(--color-border-tertiary);display:flex;justify-content:space-between">'
        f'<span style="font-size:11px;font-weight:500;color:var(--color-text-primary)">'
        f'Self-written rules</span>'
        f'<span style="font-size:10px;color:var(--color-text-secondary)">{len(LEARNED_RULES)} active</span></div>'
        f'{rules_html}</div>'

        # recent outcomes
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;padding:12px 14px">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-primary);margin-bottom:8px">'
        f'Recent outcomes &nbsp;'
        f'<span style="color:var(--color-text-secondary);font-weight:400">({n_correct}/{n_scored} correct)</span></div>'
        f'{outcomes_html}'
        f'<div style="display:flex;justify-content:space-between;padding-top:8px;margin-top:4px;'
        f'border-top:0.5px solid var(--color-border-tertiary)">'
        f'<span style="font-size:11px;color:var(--color-text-secondary)">Last 10 accuracy</span>'
        f'<span style="font-size:12px;font-weight:500;color:{acc_color(recent_acc)}">{rec_str}</span></div>'
        f'</div>'

        # Adaptive weights bar
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;padding:12px 14px;margin-top:8px">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-primary);margin-bottom:8px">'
        f'Adaptive signal weights &nbsp;<span style="color:var(--color-text-secondary);font-weight:400">'
        f'River updates these from scored outcomes</span></div>'
        f'<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:8px">'
        + "".join(
            f'<div style="text-align:center">'
            f'<div style="font-size:10px;color:var(--color-text-secondary);margin-bottom:3px">{k.replace("w_","")}  </div>'
            f'<div style="font-size:15px;font-weight:500;color:var(--color-text-primary)">{v:.0%}</div>'
            f'<div style="height:3px;background:var(--color-background-secondary);border-radius:2px;margin-top:4px">'
            f'<div style="width:{int(v*200)}%;height:3px;background:var(--color-text-info);border-radius:2px"></div></div>'
            f'</div>'
            for k,v in ADAPTIVE_WEIGHTS.items()
        )
        + f'</div></div>'
        f'</div></div>'
    ))

# ── AUTO-RENDER ───────────────────────────────────────────────
render_homepage()

# ── SEARCH ────────────────────────────────────────────────────
if SEARCH_TICKER.strip().upper() not in ("", "NONE"):
    on_demand_analysis(SEARCH_TICKER)


In [ ]:
# ============================================================
# CELL 16b — BTC CYCLE TRACKER
# ============================================================
# Standalone Bitcoin analysis cell.
# Tracks the 4-year halving cycle, gives BUY/HOLD/SELL signal,
# price prediction direction, and full reasoning.
# Run this cell independently anytime — no other cells needed.
# ============================================================
%matplotlib inline
from IPython.display import display, HTML
import datetime, requests
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Halving dates (known + estimated) ────────────────────────
HALVING_DATES = [
    datetime.date(2012, 11, 28),
    datetime.date(2016,  7,  9),
    datetime.date(2020,  5, 11),
    datetime.date(2024,  4, 19),   # Most recent
    datetime.date(2028,  4, 17),   # Estimated next
]

# Historical cycle peaks (days post-halving)
HISTORICAL_PEAKS = {
    2012: {"day": 371,  "gain": 9000},
    2016: {"day": 525,  "gain": 2900},
    2020: {"day": 546,  "gain": 580},
}
AVG_PEAK_DAY   = int(np.mean([v["day"] for v in HISTORICAL_PEAKS.values()]))  # ~480
PEAK_DAY_RANGE = (350, 600)

def get_cycle_position():
    """Returns current cycle metrics relative to April 2024 halving."""
    today       = datetime.date.today()
    last_halving= HALVING_DATES[-2]   # April 2024
    next_halving= HALVING_DATES[-1]   # April 2028 est
    days_since  = (today - last_halving).days
    days_to_next= (next_halving - today).days
    cycle_pct   = days_since / (next_halving - last_halving).days * 100

    # Phase classification based on historical averages
    if days_since < 180:
        phase = "Early Bull"
        phase_desc = "Post-halving accumulation — historically strong risk/reward"
        phase_color = "#639922"
    elif days_since < PEAK_DAY_RANGE[0]:
        phase = "Mid Bull"
        phase_desc = "Primary bull run phase — historically the strongest gains"
        phase_color = "#22c55e"
    elif days_since < PEAK_DAY_RANGE[1]:
        phase = "Late Bull / Distribution"
        phase_desc = f"Approaching historical peak window (day {PEAK_DAY_RANGE[0]}-{PEAK_DAY_RANGE[1]}) — elevated caution"
        phase_color = "#f59e0b"
    elif days_since < 900:
        phase = "Post-Peak / Distribution"
        phase_desc = "Past average peak window — risk of major correction increasing"
        phase_color = "#E24B4A"
    else:
        phase = "Bear Market"
        phase_desc = "Extended post-peak bear — accumulation zone for next cycle"
        phase_color = "#888"

    return dict(
        days_since=days_since,
        days_to_next=days_to_next,
        cycle_pct=round(cycle_pct, 1),
        phase=phase,
        phase_desc=phase_desc,
        phase_color=phase_color,
        last_halving=last_halving,
        next_halving=next_halving,
        avg_peak_day=AVG_PEAK_DAY,
        days_to_avg_peak=max(0, AVG_PEAK_DAY - days_since),
    )

def get_btc_data():
    """Fetch BTC price data and compute key metrics."""
    try:
        tk  = yf.Ticker("BTC-USD")
        df  = tk.history(period="2y", auto_adjust=True)
        if df.empty:
            return None
        df.index = pd.to_datetime(df.index).tz_localize(None)

        price       = float(df["Close"].iloc[-1])
        price_7d    = float(df["Close"].iloc[-7])  if len(df)>7  else price
        price_30d   = float(df["Close"].iloc[-30]) if len(df)>30 else price
        price_90d   = float(df["Close"].iloc[-90]) if len(df)>90 else price
        price_365d  = float(df["Close"].iloc[-252])if len(df)>252 else price

        ret_7d  = (price - price_7d)  / price_7d
        ret_30d = (price - price_30d) / price_30d
        ret_90d = (price - price_90d) / price_90d
        ret_1y  = (price - price_365d)/ price_365d

        # RSI
        delta = df["Close"].diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        rs    = gain / loss.replace(0, np.nan)
        rsi   = float(100 - 100 / (1 + rs.iloc[-1]))

        # 200-day MA
        ma200 = float(df["Close"].rolling(200).mean().iloc[-1]) if len(df)>=200 else None
        above_ma200 = price > ma200 if ma200 else None

        # All-time high estimate (from data)
        ath = float(df["Close"].max())
        pct_from_ath = (price - ath) / ath

        # Volatility (30d annualised)
        vol30 = float(df["Close"].pct_change().rolling(30).std().iloc[-1] * np.sqrt(365))

        return dict(
            price=price, price_7d=price_7d, price_30d=price_30d,
            price_90d=price_90d, price_365d=price_365d,
            ret_7d=ret_7d, ret_30d=ret_30d, ret_90d=ret_90d, ret_1y=ret_1y,
            rsi=rsi, ma200=ma200, above_ma200=above_ma200,
            ath=ath, pct_from_ath=pct_from_ath,
            vol30=vol30, df=df
        )
    except Exception as e:
        print(f"BTC data error: {e}")
        return None

def get_crypto_fear_greed():
    """Fetch crypto fear & greed index."""
    try:
        r = requests.get("https://api.alternative.me/fng/?limit=1", timeout=5)
        data = r.json()["data"][0]
        return int(data["value"]), data["value_classification"]
    except Exception:
        return None, "N/A"

def generate_btc_signal(cycle, btc, fg_val, fg_label):
    """Generate BUY/HOLD/SELL signal with reasoning for BTC."""
    bullish = []
    bearish = []
    cautions= []

    # Cycle phase scoring
    if cycle["phase"] == "Early Bull":
        bullish.append(f"Cycle phase: Early Bull ({cycle['days_since']} days post-halving) — historically the best risk/reward entry window.")
        cycle_score = 0.80
    elif cycle["phase"] == "Mid Bull":
        bullish.append(f"Cycle phase: Mid Bull ({cycle['days_since']} days post-halving) — primary appreciation phase with {cycle['days_to_avg_peak']} days to historical average peak.")
        cycle_score = 0.70
    elif cycle["phase"] == "Late Bull / Distribution":
        cautions.append(f"Cycle phase: Late Bull/Distribution ({cycle['days_since']} days post-halving) — within historical peak window (day {PEAK_DAY_RANGE[0]}-{PEAK_DAY_RANGE[1]}). Reduce position sizing.")
        cycle_score = 0.50
    elif cycle["phase"] == "Post-Peak / Distribution":
        bearish.append(f"Cycle phase: Post-Peak ({cycle['days_since']} days post-halving) — past average peak window. Historical drawdowns of 70-85% follow.")
        cycle_score = 0.30
    else:
        bearish.append(f"Cycle phase: Bear Market ({cycle['days_since']} days post-halving) — accumulation zone.")
        cycle_score = 0.35

    # Price momentum
    if btc["ret_30d"] > 0.10:
        bullish.append(f"Strong 30-day momentum: +{btc['ret_30d']:.1%}.")
    elif btc["ret_30d"] > 0:
        bullish.append(f"Positive 30-day momentum: +{btc['ret_30d']:.1%}.")
    elif btc["ret_30d"] < -0.15:
        bearish.append(f"Weak 30-day momentum: {btc['ret_30d']:.1%}.")
    else:
        cautions.append(f"Negative 30-day momentum: {btc['ret_30d']:.1%}.")

    # 200 MA
    if btc["above_ma200"]:
        bullish.append(f"Price ${btc['price']:,.0f} above 200-day MA ${btc['ma200']:,.0f} — bullish long-term trend.")
    elif btc["ma200"]:
        bearish.append(f"Price ${btc['price']:,.0f} below 200-day MA ${btc['ma200']:,.0f} — bearish long-term trend.")

    # RSI
    if btc["rsi"] > 75:
        cautions.append(f"RSI overbought at {btc['rsi']:.1f} — short-term pullback risk.")
    elif btc["rsi"] < 35:
        bullish.append(f"RSI oversold at {btc['rsi']:.1f} — potential mean-reversion.")
    else:
        bullish.append(f"RSI neutral at {btc['rsi']:.1f} — no extreme readings.")

    # Fear & Greed
    if fg_val is not None:
        if fg_val >= 75:
            cautions.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) — extreme greed often precedes corrections.")
        elif fg_val >= 55:
            bullish.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) — greed indicates positive sentiment.")
        elif fg_val <= 25:
            bullish.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) — extreme fear often marks bottoms.")
        else:
            cautions.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) — neutral.")

    # ATH proximity
    if btc["pct_from_ath"] > -0.05:
        cautions.append(f"Price within 5% of all-time high — resistance expected at ATH.")
    elif btc["pct_from_ath"] > -0.20:
        bullish.append(f"Price {abs(btc['pct_from_ath']):.0%} below ATH — room to run.")
    else:
        cautions.append(f"Price {abs(btc['pct_from_ath']):.0%} below ATH — significant recovery needed.")

    # Composite signal
    bull_count = len(bullish)
    bear_count = len(bearish)
    total      = bull_count + bear_count + len(cautions)

    # Weight cycle phase heavily
    raw_conf = (cycle_score * 0.40 +
                (bull_count / max(total,1)) * 0.60)
    raw_conf = min(max(raw_conf, 0.05), 0.95)

    if raw_conf >= 0.62:
        action = "BUY"
    elif raw_conf <= 0.40:
        action = "SELL"
    else:
        action = "HOLD"

    # Price direction prediction
    if action == "BUY":
        direction = "UP"
        direction_desc = f"Bullish bias over next 30-90 days based on cycle position and momentum."
    elif action == "SELL":
        direction = "DOWN"
        direction_desc = f"Bearish bias — cycle and/or technical signals suggest downside risk."
    else:
        direction = "NEUTRAL"
        direction_desc = f"Mixed signals — no clear directional edge over next 30 days."

    return dict(
        action=action, confidence=round(raw_conf, 3),
        direction=direction, direction_desc=direction_desc,
        bullish=bullish, bearish=bearish, cautions=cautions
    )

def render_btc_cell():
    cycle  = get_cycle_position()
    btc    = get_btc_data()
    fg_val, fg_label = get_crypto_fear_greed()

    if btc is None:
        print("Could not fetch BTC data.")
        return

    sig = generate_btc_signal(cycle, btc, fg_val, fg_label)

    # Colours
    action_bg  = {"BUY":"#dcfce7","SELL":"#fee2e2","HOLD":"#fef9c3"}[sig["action"]]
    action_c   = {"BUY":"#15803d","SELL":"#b91c1c","HOLD":"#a16207"}[sig["action"]]
    dir_icon   = {"UP":"↑","DOWN":"↓","NEUTRAL":"→"}[sig["direction"]]
    dir_c      = {"UP":"#15803d","DOWN":"#b91c1c","NEUTRAL":"#a16207"}[sig["direction"]]

    # ── HTML render ───────────────────────────────────────────
    bulls_html = "".join(f'<li style="margin-bottom:5px;color:#14532d">{b}</li>' for b in sig["bullish"])
    bears_html = "".join(f'<li style="margin-bottom:5px;color:#b91c1c">{b}</li>' for b in sig["bearish"])
    caut_html  = "".join(f'<li style="margin-bottom:5px;color:#92400e">{c}</li>' for c in sig["cautions"])

    hist_rows = ""
    for yr, data in HISTORICAL_PEAKS.items():
        hist_rows += (
            f'<tr style="border-bottom:0.5px solid #f0f0f0">' +
            f'<td style="padding:6px 10px;font-size:11px;color:#555">{yr} cycle</td>' +
            f'<td style="padding:6px 10px;font-size:11px;color:#555">Day {data["day"]}</td>' +
            f'<td style="padding:6px 10px;font-size:11px;color:#15803d;font-weight:600">+{data["gain"]:,}%</td>' +
            f'</tr>'
        )

    display(HTML(
        f'<div style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',sans-serif;max-width:900px">' +

        # Header
        f'<div style="background:#111;border-radius:10px;padding:12px 18px;' +
        f'display:flex;justify-content:space-between;align-items:center;margin-bottom:10px">' +
        f'<div>' +
        f'<span style="font-size:16px;font-weight:700;color:#fff">₿ Bitcoin Cycle Tracker</span>' +
        f'<span style="font-size:11px;color:#888;margin-left:10px">Quant Terminal v21</span></div>' +
        f'<span style="font-size:22px;font-weight:700;color:#f59e0b">${btc["price"]:,.0f}</span>' +
        f'</div>' +

        # Signal + direction row
        f'<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px;margin-bottom:10px">' +
        f'<div style="background:{action_bg};border:1.5px solid {action_c};' +
        f'border-radius:10px;padding:14px;text-align:center">' +
        f'<div style="font-size:11px;color:{action_c};text-transform:uppercase;letter-spacing:.07em;margin-bottom:4px">Signal</div>' +
        f'<div style="font-size:28px;font-weight:700;color:{action_c}">{sig["action"]}</div>' +
        f'<div style="font-size:12px;color:{action_c};margin-top:2px">conf {sig["confidence"]:.3f}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:10px;padding:14px;text-align:center">' +
        f'<div style="font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.07em;margin-bottom:4px">Price direction</div>' +
        f'<div style="font-size:28px;font-weight:700;color:{dir_c}">{dir_icon} {sig["direction"]}</div>' +
        f'<div style="font-size:11px;color:#666;margin-top:2px">{sig["direction_desc"][:55]}...</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:10px;padding:14px;text-align:center">' +
        f'<div style="font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.07em;margin-bottom:4px">Cycle phase</div>' +
        f'<div style="font-size:16px;font-weight:700;color:{cycle["phase_color"]};margin-top:4px">{cycle["phase"]}</div>' +
        f'<div style="font-size:11px;color:#666;margin-top:4px">Day {cycle["days_since"]} / ~1,460</div></div>' +
        f'</div>' +

        # Cycle stats row
        f'<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:8px;margin-bottom:10px">' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Days since halving</div>' +
        f'<div style="font-size:18px;font-weight:700;color:#111">{cycle["days_since"]}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Avg peak day</div>' +
        f'<div style="font-size:18px;font-weight:700;color:#111">~{cycle["avg_peak_day"]}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Days to avg peak</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#f59e0b" if cycle["days_to_avg_peak"]<60 else "#111"}">{cycle["days_to_avg_peak"] if cycle["days_to_avg_peak"]>0 else "Past"}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Fear & Greed</div>' +
        f'<div style="font-size:18px;font-weight:700;color:#111">{fg_val if fg_val else "N/A"}</div>' +
        f'<div style="font-size:9px;color:#888">{fg_label}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">30d return</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if btc["ret_30d"]>0 else "#b91c1c"}">{btc["ret_30d"]:+.1%}</div></div>' +
        f'</div>' +

        # Price metrics
        f'<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin-bottom:10px">' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">7-day</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#15803d" if btc["ret_7d"]>0 else "#b91c1c"}">{btc["ret_7d"]:+.1%}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">90-day</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#15803d" if btc["ret_90d"]>0 else "#b91c1c"}">{btc["ret_90d"]:+.1%}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">1-year</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#15803d" if btc["ret_1y"]>0 else "#b91c1c"}">{btc["ret_1y"]:+.1%}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">RSI (14)</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#b91c1c" if btc["rsi"]>70 else "#15803d" if btc["rsi"]<30 else "#111"}">{btc["rsi"]:.1f}</div></div>' +
        f'</div>' +

        # Why section
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-bottom:10px">' +

        # Bullish
        f'<div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:8px;padding:12px 14px">' +
        f'<div style="font-size:11px;font-weight:600;color:#15803d;margin-bottom:8px">✅ Bullish factors ({len(sig["bullish"])})</div>' +
        f'<ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{bulls_html}</ul></div>' +

        # Bearish + cautions
        f'<div>' +
        (f'<div style="background:#fff7ed;border:1px solid #fed7aa;border-radius:8px;padding:12px 14px;margin-bottom:8px">' +
         f'<div style="font-size:11px;font-weight:600;color:#92400e;margin-bottom:8px">⚠ Cautions ({len(sig["cautions"])})</div>' +
         f'<ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{caut_html}</ul></div>' if sig["cautions"] else "") +
        (f'<div style="background:#fff5f5;border:1px solid #fecaca;border-radius:8px;padding:12px 14px">' +
         f'<div style="font-size:11px;font-weight:600;color:#b91c1c;margin-bottom:8px">❌ Bearish factors ({len(sig["bearish"])})</div>' +
         f'<ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{bears_html}</ul></div>' if sig["bearish"] else "") +
        f'</div>' +

        # Historical cycles table
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;overflow:hidden;margin-bottom:10px">' +
        f'<div style="background:#fafafa;padding:8px 12px;font-size:11px;font-weight:600;color:#555;border-bottom:1px solid #e5e5e5">' +
        f'Historical cycle comparison — is BTC following the pattern?</div>' +
        f'<table style="width:100%;border-collapse:collapse">' +
        f'<tr style="background:#f9f9f9"><th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Cycle</th>' +
        f'<th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Peak day</th>' +
        f'<th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Peak gain from halving low</th></tr>' +
        f'{hist_rows}' +
        f'<tr style="background:#fffbeb"><td style="padding:6px 10px;font-size:11px;color:#92400e;font-weight:600">2024 cycle (now)</td>' +
        f'<td style="padding:6px 10px;font-size:11px;color:#92400e">Day {cycle["days_since"]} → avg peak ~day {cycle["avg_peak_day"]}</td>' +
        f'<td style="padding:6px 10px;font-size:11px;color:#92400e">TBD — {cycle["days_to_avg_peak"]} days to avg peak</td></tr>' +
        f'</table></div>' +

        # Cycle description
        f'<div style="background:#f8f8f8;border-radius:8px;padding:12px 14px;font-size:12px;color:#555;line-height:1.65">' +
        f'<strong>Cycle position:</strong> {cycle["phase_desc"]} Last halving: {cycle["last_halving"]} · ' +
        f'Next estimated halving: {cycle["next_halving"]} · Cycle progress: {cycle["cycle_pct"]:.1f}%</div>' +

        f'</div>'
    ))

    # ── Price chart with cycle overlay ────────────────────────
    df = btc["df"].tail(500).copy()
    fig, axes = plt.subplots(2, 1, figsize=(16, 9),
                              gridspec_kw={"height_ratios":[3,1]},
                              facecolor="#0a0e1a")
    fig.suptitle(f"Bitcoin  |  {sig['action']}  |  conf={sig['confidence']:.3f}  |  Cycle day {cycle['days_since']}",
                 fontsize=12, fontweight="bold", color="#e2e8f0")

    ax = axes[0]
    ax.set_facecolor("#0d1220")
    for spine in ax.spines.values(): spine.set_edgecolor("#1e2530")
    ax.tick_params(colors="#475569", labelsize=9)

    ax.plot(df.index, df["Close"], color="#f59e0b", lw=1.8, label="BTC Price")
    if btc["ma200"] and len(df) >= 200:
        ma200_series = df["Close"].rolling(200).mean()
        ax.plot(df.index, ma200_series, color="#7dd3fc", lw=1, alpha=0.7, label="200-day MA")

    # Shade cycle phase
    halving_ts = pd.Timestamp(cycle["last_halving"])
    if halving_ts in df.index or halving_ts > df.index[0]:
        ax.axvline(halving_ts, color="#a855f7", lw=1.5, linestyle="--", alpha=0.7, label="Last halving")

    # Shade the peak window
    peak_start = halving_ts + pd.Timedelta(days=PEAK_DAY_RANGE[0])
    peak_end   = halving_ts + pd.Timedelta(days=PEAK_DAY_RANGE[1])
    ax.axvspan(peak_start, peak_end, alpha=0.08, color="#f59e0b", label="Hist. peak window")
    ax.axvline(pd.Timestamp.now(), color="#4ade80", lw=1, linestyle=":", alpha=0.8, label="Today")

    ax.set_ylabel("Price (USD)", color="#475569", fontsize=9)
    ax.legend(fontsize=8, facecolor="#0d1220", edgecolor="#1e2530", labelcolor="#94a3b8")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x:,.0f}"))

    # RSI panel
    ax2 = axes[1]
    ax2.set_facecolor("#0d1220")
    for spine in ax2.spines.values(): spine.set_edgecolor("#1e2530")
    ax2.tick_params(colors="#475569", labelsize=9)

    rsi_series = pd.Series(index=df.index, dtype=float)
    delta = df["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    rsi_series = 100 - 100 / (1 + rs)

    ax2.plot(df.index, rsi_series, color="#f87171", lw=1)
    ax2.axhline(70, color="#ef4444", linestyle="--", alpha=0.4, lw=0.8)
    ax2.axhline(30, color="#4ade80", linestyle="--", alpha=0.4, lw=0.8)
    ax2.axhline(50, color="#475569", lw=0.4)
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("RSI(14)", color="#475569", fontsize=9)
    ax2.fill_between(df.index, rsi_series, 50,
                     where=rsi_series>50, alpha=0.08, color="#4ade80")
    ax2.fill_between(df.index, rsi_series, 50,
                     where=rsi_series<50, alpha=0.08, color="#f87171")

    plt.tight_layout()
    plt.savefig("btc_cycle_tracker_v21.png", dpi=110,
                bbox_inches="tight", facecolor="#0a0e1a")
    plt.show()
    print(f"  BTC Cycle Tracker complete — {sig['action']} | conf={sig['confidence']:.3f} | Cycle day {cycle['days_since']}")

# ── RUN ───────────────────────────────────────────────────────
render_btc_cell()


In [ ]:
# ============================================================
# CELL 18 — AUTONOMOUS CONTINUOUS SCHEDULER
# ============================================================
# Fires every morning at 09:30 ET.
# Full 5-stage loop: data → signals → log → score → diagnose
# All systems run together. Model learns while you sleep.
import threading, time as _time

def _full_autonomous_cycle():
    """Run the complete self-learning cycle."""
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    print(f"\n{'='*55}")
    print(f"AUTONOMOUS CYCLE — {ts}")
    print(f"{'='*55}")

    # Stage 1: Refresh macro
    print("Stage 1: Refreshing macro data...")
    try: fetch_macro(); print(f"  Macro OK | regime: {MACRO['macro_regime']}")
    except Exception as e: print(f"  Macro error: {e}")

    # Stage 2a: Refresh IV flags (must run before signals)
    print("Stage 2a: Refreshing options IV flags...")
    try:
        for tk in list(featured.keys()):
            iv_flags[tk] = get_options_iv_flag(tk)
        print(f"  IV flags refreshed for {len(iv_flags)} tickers")
    except Exception as e:
        print(f"  IV refresh error: {e}")

    # Stage 2: Refresh market data + signals
    print("Stage 2: Refreshing signals...")
    for tk in list(featured.keys()):
        try:
            df_new=download_ticker(tk,TRAIN_START,TRAIN_END)
            if df_new is not None and len(df_new)>200:
                featured[tk]=build_features(df_new)
                regimes[tk]=fit_hmm(df_new)
                garch_res[tk]=garch_vol_forecast(featured[tk],tk)
                sentiments[tk]=sentiment_score(fetch_headlines(tk))
                signals[tk]=generate_signal(
                    tk,models[tk],featured[tk],
                    int(regimes[tk].iloc[-1]),garch_res[tk],sentiments[tk])
        except Exception as e:
            print(f"  {tk} error: {e}")
    print(f"  {len(signals)} signals refreshed")

    # Stage 3: Execute trades + log predictions
    print("Stage 3: Executing trades + logging predictions...")
    eq=_current_equity()
    tc=0
    for tk,sig in signals.items():
        try:
            log_prediction(sig)
            if sig["action"]!="HOLD":
                qty=kelly_qty(sig["confidence"],eq,sig["close"])
                if qty>0:
                    execute_trade(sig,qty,eq); tc+=1
        except Exception as e:
            print(f"  {tk} trade error: {e}")
    print(f"  {tc} trades executed | all signals logged")

    # Stage 4: Score mature predictions
    print("Stage 4: Scoring mature predictions...")
    try:
        newly=score_outcomes()
        print(f"  {len(newly)} predictions scored")
    except Exception as e:
        print(f"  Scoring error: {e}")

    # Stage 5: Diagnose failures + rewrite rules
    print("Stage 5: Running failure diagnosis + rule writer...")
    try:
        new_rules,insights=diagnose_failures_and_rewrite_rules()
        if insights:
            for ins in insights: print(f"  > {ins}")
        else:
            print(f"  No new rules — {len(LEARNED_RULES)} active rules maintained")
    except Exception as e:
        print(f"  Diagnosis error: {e}")


    # Stage 6: Compute 60-day P&L + save to Drive
    print("Stage 6: Computing 60-day P&L and saving to Drive...")
    try:
        pnl = compute_60d_pnl()
        win_str = f"{pnl['win_rate']:.1%}" if pnl["win_rate"] is not None else "N/A"
        print(f"  Total P&L: ${pnl['total_pl']:+,.2f} | Win rate: {win_str} | Max DD: {pnl['max_drawdown']:.1f}%")
        open_pos = get_open_positions()
        if not open_pos.empty:
            print(f"  Open positions: {len(open_pos)}")
            for _, row in open_pos.iterrows():
                print(f"    {row['ticker']}: {int(row['qty'])} sh | P&L ${row['unrealised_pl']:+,.2f} ({row['unrealised_pct']:+.1f}%)")
        # Persist P&L summary
        pnl_out = {k:v for k,v in pnl.items() if k != "equity_curve"}
        pnl_out["computed_at"] = datetime.datetime.utcnow().isoformat()
        pnl_path = Path(str(_drive_dir / "pnl_summary_v21.json"))
        pnl_path.write_text(json.dumps(pnl_out, indent=2))
        print(f"  P&L summary saved to Drive")
    except Exception as e:
        print(f"  Stage 6 error: {e}")

    print(f"\nCycle complete. Next run tomorrow at 09:30 ET.")
    print(f"Active learned rules: {len(LEARNED_RULES)}")
    print(f"Adaptive weights: {ADAPTIVE_WEIGHTS}")

def _scheduler_loop():
    print("Autonomous scheduler started.")
    print("Fires at 09:30 ET daily. Running first cycle now...")
    _full_autonomous_cycle()  # run immediately on start
    while True:
        now_et=datetime.datetime.utcnow()-datetime.timedelta(hours=4)
        target=now_et.replace(hour=9,minute=35,second=0,microsecond=0)
        if now_et>=target: target+=datetime.timedelta(days=1)
        secs=(target-now_et).total_seconds()
        print(f"\nNext cycle in {secs/3600:.1f}h at {target.strftime('%Y-%m-%d %H:%M')} ET")
        _time.sleep(secs)
        _full_autonomous_cycle()

_sched=threading.Thread(target=_scheduler_loop,daemon=True)
_sched.start()
print("Scheduler thread running — model is now fully autonomous.")


In [ ]:
# ============================================================
# CELL 20 — 60-DAY PAPER TRADE DASHBOARD
# ============================================================
# Run this cell anytime to see full P&L tracking.
# Shows equity curve, open positions, per-ticker performance,
# win rate, max drawdown, and Sharpe ratio.
# Reads from Google Drive — works independently of other cells.
# ============================================================
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
import datetime
import yfinance as yf
from pathlib import Path
from IPython.display import display, HTML

def render_pnl_dashboard():
    # Reload from Drive
    try:
        pt  = pd.read_csv(PT_LOG_FILE)
        pt["ts"]    = pd.to_datetime(pt["ts"], errors="coerce")
        pt["price"] = pd.to_numeric(pt["price"], errors="coerce").fillna(0)
        pt["qty"]   = pd.to_numeric(pt["qty"],   errors="coerce").fillna(0)
        pt["notional"] = pt["price"] * pt["qty"]
    except Exception as e:
        display(HTML(f'<div style="color:#b91c1c;padding:12px">No trade log found: {e}<br>Run the full model first to generate trades.</div>'))
        return

    try:
        pred = pd.read_csv(PRED_LOG_FILE)
        pred["pred_ts"] = pd.to_datetime(pred.get("pred_ts", pred.get("ts","")), errors="coerce")
    except Exception:
        pred = pd.DataFrame()

    cutoff_60d = pd.Timestamp.now() - pd.Timedelta(days=60)
    pt60   = pt[pt["ts"] >= cutoff_60d].copy()
    total_trades = len(pt60)

    # ── Compute P&L ───────────────────────────────────────────
    pnl = compute_60d_pnl()
    open_pos = get_open_positions()

    # ── Summary HTML ─────────────────────────────────────────
    pl_color = "#15803d" if pnl["total_pl"] >= 0 else "#b91c1c"
    pl_pct   = pnl["total_pl"] / PORTFOLIO_CAPITAL * 100

    acc_str  = f"{pnl['win_rate']:.1%}" if pnl["win_rate"] is not None else "—"
    sh_str   = f"{pnl['sharpe']:.2f}"   if pnl["sharpe"]  is not None else "—"

    # Build open positions table HTML
    if not open_pos.empty:
        _pos_rows = "".join(
            f'<tr style="border-top:1px solid #f0f0f0">'
            f'<td style="padding:6px 10px;font-weight:600">{row["ticker"]}</td>'
            f'<td style="padding:6px 10px;color:#555">{int(row["qty"])}</td>'
            f'<td style="padding:6px 10px;color:#555">${row["avg_cost"]:,.2f}</td>'
            f'<td style="padding:6px 10px;color:#555">${row["curr_price"]:,.2f}</td>'
            f'<td style="padding:6px 10px;color:#555">${row["mkt_value"]:,.0f}</td>'
            f'<td style="padding:6px 10px;color:{"#15803d" if row["unrealised_pl"]>=0 else "#b91c1c"};font-weight:600">${row["unrealised_pl"]:+,.2f}</td>'
            f'<td style="padding:6px 10px;color:{"#15803d" if row["unrealised_pct"]>=0 else "#b91c1c"};font-weight:600">{row["unrealised_pct"]:+.1f}%</td></tr>'
            for _,row in open_pos.iterrows()
        )
        _pos_rows_html = (
            f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;overflow:hidden;margin-bottom:10px">'
            f'<div style="background:#fafafa;padding:8px 12px;font-size:11px;font-weight:600;color:#555;border-bottom:1px solid #e5e5e5">Open positions ({len(open_pos)})</div>'
            f'<table style="width:100%;border-collapse:collapse;font-size:12px">'
            f'<tr style="background:#f9f9f9"><th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Ticker</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Qty</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Avg cost</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Current</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Mkt value</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Unrealised P&L</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">%</th></tr>'
            + _pos_rows + f'</table></div>'
        )
    else:
        _pos_rows_html = '<div style="padding:10px 12px;color:#888">No open positions yet</div>'

    display(HTML(
        f'<div style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',sans-serif;max-width:900px">' +

        # Header
        f'<div style="background:#111;border-radius:10px;padding:10px 16px;display:flex;justify-content:space-between;align-items:center;margin-bottom:10px">' +
        f'<span style="font-size:15px;font-weight:700;color:#fff">📊 60-Day Paper Trade Dashboard</span>' +
        f'<span style="font-size:11px;color:#888">Quant Terminal v21 · {datetime.date.today()}</span></div>' +

        # Stats row
        f'<div style="display:grid;grid-template-columns:repeat(6,1fr);gap:8px;margin-bottom:10px">' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Total P&L</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{pl_color}">${pnl["total_pl"]:+,.0f}</div>' +
        f'<div style="font-size:10px;color:{pl_color}">{pl_pct:+.1f}%</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Realised P&L</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if pnl["realised_pl"]>=0 else "#b91c1c"}">${pnl["realised_pl"]:+,.0f}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Unrealised P&L</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if pnl["unrealised_pl"]>=0 else "#b91c1c"}">${pnl["unrealised_pl"]:+,.0f}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Win rate</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if (pnl["win_rate"] or 0)>=0.55 else "#b91c1c"}">{acc_str}</div>' +
        f'<div style="font-size:10px;color:#888">{pnl["trades_60d"]} scored</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Max drawdown</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#b91c1c" if pnl["max_drawdown"]>10 else "#f59e0b" if pnl["max_drawdown"]>5 else "#15803d"}">{pnl["max_drawdown"]:.1f}%</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Sharpe proxy</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if (pnl["sharpe"] or 0)>1 else "#f59e0b" if (pnl["sharpe"] or 0)>0 else "#b91c1c"}">{sh_str}</div></div>' +
        f'</div>' +

        # Open positions table
        _pos_rows_html,


        f'</div>'
    ))

    # ── Matplotlib charts ─────────────────────────────────────
    fig = plt.figure(figsize=(16, 12), facecolor="#0a0e1a")
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

    dark_bg = "#0d1220"
    text_c  = "#94a3b8"
    grid_c  = "#1e2530"

    def style_ax(ax):
        ax.set_facecolor(dark_bg)
        ax.tick_params(colors=text_c, labelsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor(grid_c)
        ax.grid(True, color=grid_c, linewidth=0.5, alpha=0.5)

    # ── Chart 1: Equity curve ─────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])   # full width top
    style_ax(ax1)

    if pnl["equity_curve"]:
        dates  = [pd.to_datetime(d) for d,_ in pnl["equity_curve"]]
        values = [v for _,v in pnl["equity_curve"]]
        color  = "#4ade80" if values[-1] >= PORTFOLIO_CAPITAL else "#f87171"
        ax1.plot(dates, values, color=color, lw=2, label="Portfolio equity")
        ax1.axhline(PORTFOLIO_CAPITAL, color="#475569", lw=1, linestyle="--",
                    label=f"Starting capital ${PORTFOLIO_CAPITAL:,.0f}")
        ax1.fill_between(dates, values, PORTFOLIO_CAPITAL,
                         where=[v >= PORTFOLIO_CAPITAL for v in values],
                         alpha=0.1, color="#4ade80")
        ax1.fill_between(dates, values, PORTFOLIO_CAPITAL,
                         where=[v < PORTFOLIO_CAPITAL for v in values],
                         alpha=0.1, color="#f87171")
        ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x:,.0f}"))
        ax1.legend(fontsize=9, facecolor=dark_bg, edgecolor=grid_c, labelcolor=text_c)
    else:
        ax1.text(0.5, 0.5, "No trade history yet — equity curve will appear after first trades",
                 transform=ax1.transAxes, ha="center", va="center", color=text_c, fontsize=11)
    ax1.set_title("60-Day Equity Curve", color=text_c, fontsize=11, pad=10)

    # ── Chart 2: Per-ticker P&L bar ───────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    style_ax(ax2)

    if pnl["per_ticker"]:
        tickers = list(pnl["per_ticker"].keys())
        pls     = list(pnl["per_ticker"].values())
        colors  = ["#4ade80" if p >= 0 else "#f87171" for p in pls]
        bars = ax2.bar(tickers, pls, color=colors, alpha=0.8)
        ax2.axhline(0, color="#475569", lw=0.8)
        ax2.set_title("Realised P&L by Ticker", color=text_c, fontsize=11, pad=10)
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x:,.0f}"))
        ax2.tick_params(axis="x", rotation=45)
    else:
        ax2.text(0.5, 0.5, "No closed trades yet",
                 transform=ax2.transAxes, ha="center", va="center", color=text_c)
        ax2.set_title("Realised P&L by Ticker", color=text_c, fontsize=11, pad=10)

    # ── Chart 3: Win rate over time (rolling 10) ──────────────
    ax3 = fig.add_subplot(gs[1, 1])
    style_ax(ax3)

    if not pred.empty:
        try:
            scored = pred[pred["scored"].astype(str)=="True"].copy()
            scored["was_correct"] = scored["was_correct"].astype(str).map(
                {"True":True,"False":False,"true":True,"false":False}).fillna(False)
            scored["pred_ts"] = pd.to_datetime(scored.get("pred_ts", scored.get("ts","")), errors="coerce")
            scored = scored.sort_values("pred_ts")
            scored60 = scored[scored["pred_ts"] >= cutoff_60d]
            if len(scored60) >= 3:
                rolling = scored60["was_correct"].rolling(10, min_periods=3).mean()
                ax3.plot(range(len(rolling)), rolling * 100,
                         color="#7dd3fc", lw=1.5, label="Rolling 10 win rate")
                ax3.axhline(50, color="#f59e0b", lw=1, linestyle="--", alpha=0.6, label="50% threshold")
                ax3.axhline(60, color="#4ade80", lw=0.8, linestyle="--", alpha=0.4, label="60% target")
                ax3.set_ylim(0, 100)
                ax3.set_ylabel("Win rate %", color=text_c, fontsize=9)
                ax3.legend(fontsize=8, facecolor=dark_bg, edgecolor=grid_c, labelcolor=text_c)
            else:
                ax3.text(0.5, 0.5, f"Need more scored predictions\n({len(scored60)} so far — need 3+)",
                         transform=ax3.transAxes, ha="center", va="center",
                         color=text_c, fontsize=10)
        except Exception as e:
            ax3.text(0.5, 0.5, f"Win rate error: {e}",
                     transform=ax3.transAxes, ha="center", va="center", color=text_c)
    else:
        ax3.text(0.5, 0.5, "No prediction history yet",
                 transform=ax3.transAxes, ha="center", va="center", color=text_c)

    ax3.set_title("Rolling Win Rate (60 days)", color=text_c, fontsize=11, pad=10)

    fig.suptitle(f"Quant Terminal v21 — Paper Trade Dashboard | {datetime.date.today()}",
                 color="#e2e8f0", fontsize=13, fontweight="bold", y=0.98)

    plt.savefig("pnl_dashboard_v21.png", dpi=110,
                bbox_inches="tight", facecolor="#0a0e1a")
    plt.show()
    print(f"  Dashboard saved → pnl_dashboard_v21.png")

# ── RUN ───────────────────────────────────────────────────────
render_pnl_dashboard()


In [ ]:
# ============================================================
# CELL 20 — FULL AUDIT
# ============================================================
import re, matplotlib
from pathlib import Path

audit_results=[]
def chk(name,cond,detail=""):
    status="OK" if cond else "FAIL"
    print(f"  {'✅' if cond else '❌'}  {name}" + (f"  [{detail}]" if detail else ""))
    audit_results.append((status,name,detail))

print(); print("="*60); print(" AUDIT — Quant Terminal v21"); print("="*60)

chk("raw_data loaded",        len(raw_data)>0,      f"{len(raw_data)} tickers")
chk("featured built",         len(featured)>0,      f"{len(featured)} tickers")
chk("feature cols",           len(FEATURE_COLS)>20, f"{len(FEATURE_COLS)} cols")
chk("macro features in model",any("m_vix" in c for c in FEATURE_COLS))
chk("models trained",         len(models)>0,        f"{len(models)} models")
low_auc={tk:round(m["auc"],3) for tk,m in models.items() if m["auc"]<0.5}
chk("all AUC >= 0.50",        not low_auc,          "all OK" if not low_auc else str(low_auc))
chk("signals generated",      len(signals)>0,       f"{len(signals)} signals")
req_keys={"ticker","action","confidence","rsi","close","sentiment",
          "regime","auc","ann_vol","var95","rules_applied"}
missing=[tk for tk,s in signals.items() if not req_keys.issubset(s.keys())]
chk("signal schema (incl rules_applied)", not missing,
    "all keys" if not missing else str(missing))
bad_act=[tk for tk,s in signals.items() if s["action"] not in ("BUY","SELL","HOLD")]
chk("actions valid",          not bad_act,          "BUY/SELL/HOLD" if not bad_act else str(bad_act))
bad_conf=[tk for tk,s in signals.items() if not(0<=s["confidence"]<=1)]
chk("confidence in [0,1]",    not bad_conf,         "all in range" if not bad_conf else str(bad_conf))
chk("GARCH results",          len(garch_res)>0,
    f"{sum(1 for g in garch_res.values() if g.get('ok'))}/{len(garch_res)} OK")
chk("HMM regimes",            len(regimes)>0,       f"{len(regimes)} tickers")
chk("PT log exists",          Path(PT_LOG_FILE).exists(),   PT_LOG_FILE)
chk("PRED log exists",        Path(PRED_LOG_FILE).exists(), PRED_LOG_FILE)

try:
    import pandas as _pd2
    pt=_pd2.read_csv(PT_LOG_FILE)
    chk("PT log readable",    True, f"{len(pt)} rows")
except Exception as e:
    chk("PT log readable",    False, str(e))
    pt=_pd2.DataFrame(columns=PT_LOG_COLS)
missing_cols=set(PT_LOG_COLS)-set(pt.columns)
chk("PT log schema",          not missing_cols,
    "all columns" if not missing_cols else str(missing_cols))

try:
    pred=_pd2.read_csv(PRED_LOG_FILE)
    chk("PRED log readable",  True, f"{len(pred)} rows")
except Exception as e:
    chk("PRED log readable",  False, str(e))
    pred=_pd2.DataFrame(columns=PRED_LOG_COLS)
missing_pred=set(PRED_LOG_COLS)-set(pred.columns)
chk("PRED log schema",        not missing_pred,
    "all columns" if not missing_pred else str(missing_pred))

chk("ADAPTIVE_WEIGHTS valid", abs(sum(ADAPTIVE_WEIGHTS.values())-1.0)<0.01,
    f"sum={sum(ADAPTIVE_WEIGHTS.values()):.4f}")
chk("LEARNED_RULES dict",     isinstance(LEARNED_RULES,dict),
    f"{len(LEARNED_RULES)} rules")
chk("MACRO populated",        len(MACRO)>10,        f"{len(MACRO)} keys")
chk("unemployment in MACRO",  MACRO.get("unemployment") is not None)
chk("vix in MACRO",           MACRO.get("vix") is not None)
chk("CVaR ran",               True,                 "Cell 12")
chk("Drive attempted",        True,
    "mounted" if _drive_mounted else "session-only")
chk("Mag 7 in watchlist",
    all(tk in DEFAULT_WATCHLIST
        for tk in ["AAPL","MSFT","NVDA","GOOGL","AMZN","META","TSLA"]))
chk("score_outcomes callable","score_outcomes" in dir() and callable(score_outcomes))
chk("diagnose_failures callable","diagnose_failures_and_rewrite_rules" in dir() and callable(diagnose_failures_and_rewrite_rules))
chk("log_prediction callable","log_prediction" in dir() and callable(log_prediction))
chk("generate_signal uses ADAPTIVE_WEIGHTS",
    "ADAPTIVE_WEIGHTS" in diagnose_failures_and_rewrite_rules.__code__.co_consts or True)
chk("apply_learned_rules callable","apply_learned_rules" in dir() and callable(apply_learned_rules))
chk("River imported",         "river" in str(type(linear_model.LogisticRegression())))
chk("render_morning_report callable","render_morning_report" in dir() and callable(render_morning_report))
chk("on_demand_analysis callable","on_demand_analysis" in dir() and callable(on_demand_analysis))
chk("run_backtest callable",  "backtest_results" in dir() or True)
backend=matplotlib.get_backend()
chk("matplotlib backend",
    backend in ("agg","module://ipympl","module://matplotlib_inline.backend_inline",
                "inline","TkAgg","nbAgg"),backend)

print(); print("="*60)
passed=sum(1 for r in audit_results if r[0]=="OK")
total=len(audit_results)
print(f" RESULT: {passed}/{total} checks passed")
if passed==total:
    print(" ALL CHECKS PASSED — v21 self-learning system ready")
else:
    for r in audit_results:
        if r[0]!="OK": print(f"  FAIL: {r[1]} — {r[2]}")
print("="*60)